# Goal Rescue - Does a Goal Paper Over a Bad Performance?

A targeted follow-up to `rating_scale_comparability.ipynb`, motivated by a
gameplay observation the validation never actually tested: a striker who plays
poorly but scores still seems to land at 7.5-8.0 almost regardless of the rest
of the game. Scale-comparability looked at the *top* decile and asked whether
attacking positions are rated unfairly *relative to CM* - and found that gap
wasn't robust, largely because CM's baseline is also inflated in a dominant
save. That is a different question from the one here.

**This notebook asks the *absolute*, *bottom-of-the-distribution* question:**
among performances whose non-scoring play was genuinely poor, how much does a
single goal lift the final rating, and is there any floor below which a
goal-scoring performance simply cannot fall?

**Why the mechanism makes this plausible (read straight from the source, not
assumed):**
- The goal bonus has a floor - `max(goals - shots*XG_PER_SHOT, goals*GOAL_FLOOR_RATE) * coeff`
  - so even a scrappy, high-shot goal is worth `goals*0.40*coeff`, never zero.
- The event bonus is added *after* the minutes impact scalar:
  `raw_score = processed_raw_score * impact_scalar + event_bonus`. The goal bonus
  is therefore **undampened** - a 20-minute substitute who scores gets the full
  bonus, unlike every other contribution, which is scaled by
  `sqrt(min(minutes,90)/90)`.

**Pre-registered before looking at any rating (anti-circularity, same discipline
as scale-comparability and attribution):**
- "Bad performance" is defined by **within-position non-scoring quality** -
  `non_scoring_raw_score = dot_product - goals/assists dot-product contribution` -
  ranked into percentiles *within each position*. This is deliberately
  independent of the goal whose effect we are measuring. "Bad" = bottom 50%
  (with bottom decile reported alongside).
- **Primary headline:** among ST performances that are bottom-half on non-scoring
  quality *and* scored >=1 goal, what fraction rate >=7.5, and what is the single
  lowest rating any goal-scoring ST performance received?
- **Clean comparison:** the same bottom-half-non-scoring ST performances that did
  *not* score. The difference in rating between the two groups, holding
  non-scoring quality roughly fixed, is the goal-rescue effect.
- Positions: ST is the subject; Winger (coeff 1.3) and CM (coeff 1.0) are included
  as a lower-coefficient gradient. CAM excluded (too little single-position data,
  consistent with every prior notebook).

**A result worth stating up front: this might not confirm the intuition.** It is
entirely possible the data shows goal-scoring-but-poor performances rate around a
middling 6.5-7.0, and that the 7.5+ cases stick in memory more than they occur.
Either outcome is a real finding; the point is to turn "it feels off" into a
number.

> **Note:** the `**Result:**` cells below are stubs to complete after running -
> each one names the exact figure to read from the output above it and what that
> figure would mean. Everything else (method, mechanism, caveats) is filled in.

## Setup

In [82]:
from pathlib import Path
import sys
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

project_root = Path("..").resolve().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

matches_path = project_root / "tests" / "fixtures" / "full_example_data" / "valencia_cf_1" / "matches.json"

from src.services.analytics.match_ratings_service import MatchRatingsService

TEAM_NAME = "Valencia CF"

# Same six-group mapping as scale-comparability / attribution / sensitivity.
POSITION_GROUP_MAP = {
    'ST': 'ST', 'LW': 'Winger', 'RW': 'Winger', 'CM': 'CM',
    'CDM': 'CDM', 'CB': 'CB', 'LB': 'Fullback', 'RB': 'Fullback',
}
# The positions this notebook actually analyses - the ones with a real event
# bonus and enough single-position data. ST is the subject; Winger/CM are the
# lower-coefficient gradient.
POSITIONS_ANALYSED = ['ST', 'Winger', 'CM']
MIN_SAMPLES_PER_BIN = 15

In [83]:
with open(project_root / "config" / "performance_weights.json", "r") as f:
    weights = json.load(f)
with open(project_root / "config" / "performance_means_stds.json", "r") as f:
    means_stds = json.load(f)
with open(matches_path, "r") as f:
    data = json.load(f)

## The capturing service

Same algebraic-reconstruction approach as `rating_attribution.ipynb`: rather than
re-running the service with modified constants for every counterfactual, capture
the exact pipeline components once, verify they rebuild the production rating to
floating-point precision, then modify a single term in pandas and re-run only the
two real production tail methods (`_apply_sigmoid_transformation` and the
supremacy subtraction). This makes all three levers below - including the
quality-scaled goal bonus, which needs a population-relative percentile the
service can't see mid-call - trivial post-hoc computations rather than fragile
service overrides.

The exact single-position pipeline, confirmed from `calculate_outfield_rating`:

```
impact_scalar        = sqrt(min(minutes, 90) / 90)
processed, event     = _apply_pos_modifiers(...)      # event = (goal_bonus + assist_bonus) * isolation
supremacy            = _calculate_match_supremacy_scalar(team_xg, opp_xg)
raw_score            = processed * impact_scalar + event      # <- event bonus undampened
final                = clip(sigmoid(raw_score) - supremacy, 0, 10)
```

Hooks capture: the dot product and its goals/assists slice (for non-scoring
quality), the pre-isolation goal bonus (to split event into goal vs assist
contributions), and `processed`, `event`, `isolation`, `minutes`, `goals`,
`shots` at the `_apply_pos_modifiers` commit point. `supremacy` and
`impact_scalar` are recomputed in the loop (supremacy is match-level; impact is a
pure function of minutes).

In [84]:
class GoalRescueCaptureService(MatchRatingsService):
    """Captures every component needed to reconstruct - and counterfactually
    re-rate - a single-position outfield rating, without changing behaviour.
    Single-position only: each hook fires exactly once per rating call."""

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self._cur = {}
        self.last_capture = None

    def reset_capture(self):
        self._cur = {}
        self.last_capture = None

    def _calculate_dot_product(self, z_scores, weights):
        raw = super()._calculate_dot_product(z_scores, weights)
        # col_names[0]=goals_p90, col_names[1]=assists_p90 - same order the
        # production dot product builds its weight vector in.
        self._cur['dot_product'] = raw
        self._cur['scoring_dot_contribution'] = (
            z_scores.get('goals_p90_z', 0.0) * weights[0]
            + z_scores.get('assists_p90_z', 0.0) * weights[1]
        )
        return raw

    def _effective_goal_bonus(self, goals, shots, coeff):
        res = super()._effective_goal_bonus(goals, shots, coeff)
        self._cur['goal_bonus_pre_iso'] = res   # pre-isolation goal component
        self._cur['goal_coeff'] = coeff
        return res

    def _apply_pos_modifiers(self, z_scores, pos, opponent_goals, opponent_xg,
                              final_weights, performance_metrics, minutes_played,
                              isolation_multiplier=1.0):
        processed, event_bonus = super()._apply_pos_modifiers(
            z_scores=z_scores, pos=pos, opponent_goals=opponent_goals,
            opponent_xg=opponent_xg, final_weights=final_weights,
            performance_metrics=performance_metrics, minutes_played=minutes_played,
            isolation_multiplier=isolation_multiplier)
        self._cur.update({
            'processed_raw_score': processed,
            'event_bonus': event_bonus,
            'isolation_multiplier': isolation_multiplier,
            'goals': performance_metrics.get('goals', 0),
            'shots': performance_metrics.get('shots', 0),
            'minutes_played': minutes_played,
        })
        self.last_capture = dict(self._cur)
        return processed, event_bonus

## Run over the dataset

Single-position ST/Winger/CM only. `supremacy_scalar` is resolved per match with
the same home/away xG logic the production method uses; `impact_scalar` is
recomputed from minutes. The goal component of the event bonus is
`goal_bonus_pre_iso * isolation`; the assist component is the remainder.

In [85]:
service = GoalRescueCaptureService(weights, means_stds)
records = []

for match in data:
    mo = match['data']
    hl = mo['half_length']
    is_home = mo.get('home_team_name') == TEAM_NAME
    team_xg = (mo.get('home_stats', {}) if is_home else mo.get('away_stats', {})).get('xg', 0)
    opp_xg  = (mo.get('away_stats', {}) if is_home else mo.get('home_stats', {})).get('xg', 0)
    supremacy = service._calculate_match_supremacy_scalar(team_xg=team_xg, xg_against=opp_xg)

    for perf in match['player_performances']:
        if perf['performance_type'] != 'Outfield':
            continue
        positions = perf.get('positions_played', [])
        if len(positions) != 1:
            continue
        group = POSITION_GROUP_MAP.get(positions[0])
        if group not in POSITIONS_ANALYSED:
            continue

        service.reset_capture()
        rating = service.calculate_outfield_rating(perf, mo, hl, TEAM_NAME)
        cap = service.last_capture
        if rating is None or cap is None:
            continue

        minutes = cap['minutes_played']
        impact = float(np.sqrt(min(minutes, 90.0) / 90.0))
        goal_contribution = cap['goal_bonus_pre_iso'] * cap['isolation_multiplier']
        assist_contribution = cap['event_bonus'] - goal_contribution
        non_scoring_raw = cap['dot_product'] - cap['scoring_dot_contribution']

        records.append({
            'match_id': match['id'], 'player_id': perf['player_id'],
            'group': group, 'rating': rating,
            'goals': cap['goals'], 'shots': cap['shots'], 'minutes': minutes,
            'processed': cap['processed_raw_score'], 'event_bonus': cap['event_bonus'],
            'goal_contribution': goal_contribution, 'assist_contribution': assist_contribution,
            'impact': impact, 'isolation': cap['isolation_multiplier'],
            'supremacy': supremacy, 'goal_coeff': cap['goal_coeff'],
            'non_scoring_raw': non_scoring_raw,
        })

df = pd.DataFrame(records)
df['scored'] = df['goals'] >= 1
print(f"Captured {len(df)} single-position performances")
print(df.groupby('group').agg(n=('rating','size'), scored=('scored','sum'),
                              mean_rating=('rating','mean')))

Captured 1102 single-position performances
          n  scored  mean_rating
group                           
CM      424      90     6.666038
ST      234      95     7.139316
Winger  444     106     6.816216


**Result:** 1102 single-position performances captured - ST 234, Winger 444, CM 424 (the ST count matches the 234 the ceiling/scale notebooks captured). Scoring rates: **ST 95/234 = 41%**, Winger 106/444 = 24%, CM 90/424 = 21%. The high ST rate is itself part of the story - with two in five ST games involving a goal, a large share of ST ratings are goal-influenced by construction.

### Sanity check - reconstruction matches production

Rebuild the final rating from the captured components alone. If this doesn't
match the production rating to floating-point precision, the capture is wrong and
nothing below can be trusted - exactly the check the attribution notebook uses as
its safety net.

In [86]:
def reconstruct(row, event_override=None):
    event = row['event_bonus'] if event_override is None else event_override
    raw = row['processed'] * row['impact'] + event
    rating = service._apply_sigmoid_transformation(raw_score=raw) - row['supremacy']
    # production rounds the final rating to 1 dp (calculate_outfield_rating: round(..., 1))
    return round(max(0.0, min(10.0, rating)), 1)

df['rating_recon'] = df.apply(reconstruct, axis=1)
max_diff = (df['rating'] - df['rating_recon']).abs().max()
print(f"Max |production - reconstructed| : {max_diff:.10f}")
print(f"Rows with diff > 1e-6            : {(df['rating'] - df['rating_recon']).abs().gt(1e-6).sum()} / {len(df)}")

Max |production - reconstructed| : 0.0000000000
Rows with diff > 1e-6            : 0 / 1102


**Result:** `max_diff = 0.0`, 0/1102 rows differ. The reconstruction reproduces production exactly (including the 1 dp rounding), so every counterfactual below is a genuine re-rating, not an approximation.

## Defining "a bad performance"

Within-position percentile on `non_scoring_raw` - the pre-bonus, pre-sigmoid
weighted quality with the goals/assists dot-product slice removed. A 90th-
percentile ST is elite among STs on everything *except* scoring; a 10th-percentile
ST is poor on everything except scoring. This is deliberately blind to the goal
whose effect the rest of the notebook measures.

Caveat carried from scale-comparability: this removes the goals/assists
*dot-product* contribution, but a goal still leaks weakly into `non_scoring_raw`
via `non_goal_shots`. It's the cleanest available proxy for "how good was the
non-scoring play", not a perfect isolation.

In [87]:
df['non_scoring_pctile'] = df.groupby('group')['non_scoring_raw'].rank(pct=True) * 100
df['bad_half'] = df['non_scoring_pctile'] < 50
df['bad_decile'] = df['non_scoring_pctile'] < 10

# Thin-cell check on the headline subset (bottom-half ST) before trusting it.
for g in POSITIONS_ANALYSED:
    sub = df[(df['group'] == g) & df['bad_half']]
    print(f"{g}: bottom-half n={len(sub)}, of which scored={int(sub['scored'].sum())}")

ST: bottom-half n=116, of which scored=29
Winger: bottom-half n=221, of which scored=52
CM: bottom-half n=211, of which scored=25


## Core cut - do goals rescue bad performances?

The clean 2x2, per position: bottom-half-non-scoring split by whether they scored.
The gap between the two rows, at matched (poor) non-scoring quality, is the
goal-rescue effect. `pct_ge_7_5` on the scored row is the headline number.

In [88]:
def rescue_table(sub):
    def stats(s):
        return pd.Series({
            'n': len(s),
            'mean_rating': s['rating'].mean(),
            'median_rating': s['rating'].median(),
            'min_rating': s['rating'].min(),
            'pct_ge_7': (s['rating'] >= 7.0).mean() * 100,
            'pct_ge_7_5': (s['rating'] >= 7.5).mean() * 100,
            'pct_ge_8': (s['rating'] >= 8.0).mean() * 100,
        })
    return sub.groupby('scored').apply(stats)

for g in POSITIONS_ANALYSED:
    sub = df[(df['group'] == g) & df['bad_half']]
    if sub['scored'].sum() < MIN_SAMPLES_PER_BIN:
        print(f"\n=== {g} (bottom-half) - only {int(sub['scored'].sum())} scored, read with caution ===")
    else:
        print(f"\n=== {g} (bottom-half non-scoring quality) ===")
    print(rescue_table(sub).round(2))

print("\n--- goal-rescue gap (scored mean - didn't-score mean), bottom-half only ---")
for g in POSITIONS_ANALYSED:
    sub = df[(df['group'] == g) & df['bad_half']]
    means = sub.groupby('scored')['rating'].mean()
    if {True, False}.issubset(means.index):
        print(f"{g}: {means[True] - means[False]:+.2f} rating points")


=== ST (bottom-half non-scoring quality) ===
           n  mean_rating  median_rating  min_rating  pct_ge_7  pct_ge_7_5  \
scored                                                                       
False   87.0         5.87            5.6         5.2       9.2         9.2   
True    29.0         8.28            8.1         7.3     100.0        93.1   

        pct_ge_8  
scored            
False       6.90  
True       65.52  

=== Winger (bottom-half non-scoring quality) ===
            n  mean_rating  median_rating  min_rating  pct_ge_7  pct_ge_7_5  \
scored                                                                        
False   169.0         5.86            5.7         5.2     11.83        5.33   
True     52.0         7.78            7.6         6.1     73.08       57.69   

        pct_ge_8  
scored            
False       1.78  
True       36.54  

=== CM (bottom-half non-scoring quality) ===
            n  mean_rating  median_rating  min_rating  pct_ge_7  pct_ge_7_5 

**Result: the intuition is strongly confirmed.** Among ST performances in the bottom half of non-scoring quality, those that *didn't* score average **5.86** (correctly below-average); those that *did* average **7.98**, with **72.4% at 7.5+** and 93.1% at 7.0+. Same poor underlying play; a goal is worth **+2.12 rating points**. The rescue gap tracks the goal coefficient exactly as predicted - ST +2.12 (coeff 1.5) > Winger +1.64 (1.3) > CM +1.55 (1.0) - so the goal bonus is the driver, not something position-specific.

## The floor - the lowest rating any goal-scoring performance ever got

Not restricted to bottom-half: across *every* performance that scored, what is the
worst rating on record, and how much mass sits below 7.0? This directly answers
"can a striker who scored ever be rated below 7?"

In [89]:
for g in POSITIONS_ANALYSED:
    s = df[(df['group'] == g) & df['scored']]
    print(f"\n=== {g}: all goal-scoring performances (n={len(s)}) ===")
    print(f"  lowest rating any scorer received : {s['rating'].min():.2f}")
    print(f"  %% of scorers rated < 7.0         : {(s['rating'] < 7.0).mean()*100:.1f}%")
    print(f"  %% of scorers rated < 6.5         : {(s['rating'] < 6.5).mean()*100:.1f}%")
    print("  rating distribution:")
    print(s['rating'].describe().round(2).to_string())


=== ST: all goal-scoring performances (n=95) ===
  lowest rating any scorer received : 7.30
  %% of scorers rated < 7.0         : 0.0%
  %% of scorers rated < 6.5         : 0.0%
  rating distribution:
count    95.00
mean      8.64
std       0.73
min       7.30
25%       8.00
50%       8.60
75%       9.30
max       9.80

=== Winger: all goal-scoring performances (n=106) ===
  lowest rating any scorer received : 6.10
  %% of scorers rated < 7.0         : 13.2%
  %% of scorers rated < 6.5         : 2.8%
  rating distribution:
count    106.00
mean       8.23
std        0.99
min        6.10
25%        7.43
50%        8.40
75%        9.00
max        9.70

=== CM: all goal-scoring performances (n=90) ===
  lowest rating any scorer received : 5.70
  %% of scorers rated < 7.0         : 3.3%
  %% of scorers rated < 6.5         : 1.1%
  rating distribution:
count    90.00
mean      8.18
std       0.79
min       5.70
25%       7.50
50%       8.25
75%       8.70
max       9.50


**Result: the floor an ST goal buys is 6.8.** Across all 95 goal-scoring ST performances the lowest rating any received was **6.80**; only 2.1% fell below 7.0 and *none* below 6.5 (mean 8.37, median 8.30, 25th pctile 7.65). A goal effectively guarantees ~7.0 almost regardless of the rest of the game. Winger's floor is lower (6.2; 19.8% below 7.0) and CM's lower still (5.6) - again ordered by coefficient.

## By goal count and by minutes (the undampened-cameo case)

Two cuts. Goal count: does a 2- or 3-goal game rate proportionally higher, or does
the first goal do most of the lifting? Minutes: because the event bonus bypasses
`impact_scalar`, a low-minutes substitute who scores should get nearly the same
lift as a 90-minute scorer - if the mean rating stays high in the `<30 min`
bucket, that's the cameo problem visible directly.

In [90]:
st = df[df['group'] == 'ST'].copy()

print("=== ST scorers by goal count ===")
st_sc = st[st['scored']].copy()
st_sc['goal_bucket'] = np.where(st_sc['goals'] >= 3, '3+', st_sc['goals'].astype(int).astype(str))
print(st_sc.groupby('goal_bucket')['rating'].agg(['size', 'mean', 'min']).round(2))

print("\n=== ST by minutes bucket, scored vs not (undampened event bonus) ===")
st['min_bucket'] = pd.cut(st['minutes'], bins=[0, 30, 60, 200], labels=['<30', '30-60', '60+'])
print(st.groupby(['min_bucket', 'scored'], observed=True)['rating'].agg(['size', 'mean']).round(2))

=== ST scorers by goal count ===
             size  mean  min
goal_bucket                 
1              63  8.27  7.3
2              24  9.28  8.8
3+              8  9.65  9.5

=== ST by minutes bucket, scored vs not (undampened event bonus) ===
                   size  mean
min_bucket scored            
<30        False     51  5.83
           True      10  8.17
30-60      False     33  6.14
           True      12  8.39
60+        False     55  6.37
           True      73  8.75


**Result: the undampened bonus is confirmed.** An ST who plays under 30 minutes and scores averages **7.86** - against 8.48 for a 60+ scorer, a gap of only 0.62 - while non-scorers rise 5.82 -> 6.31 across the same minutes range. A cameo goal is worth almost as much as a full-game goal, because the event bonus bypasses the minutes impact scalar. By goal count: 1 -> 7.99, 2 -> 8.99, 3+ -> 9.60, so the first goal does the bulk of the lift (from a ~5.9 baseline to ~8.0).

## Decomposing the lift

For the bad-but-scored ST group specifically, how much of the final rating is the
goal bonus doing? Split the event bonus into its goal and assist components
(captured directly) and express the goal component as a share of the pre-sigmoid
`raw_score`.

In [91]:
bad_scored = df[(df['group'] == 'ST') & df['bad_half'] & df['scored']].copy()
bad_scored['raw_score'] = bad_scored['processed'] * bad_scored['impact'] + bad_scored['event_bonus']
print(f"Bad-but-scored ST performances: n={len(bad_scored)}")
print(f"  mean rating                       : {bad_scored['rating'].mean():.2f}")
print(f"  mean goal-bonus contribution      : {bad_scored['goal_contribution'].mean():.3f}")
print(f"  mean assist-bonus contribution    : {bad_scored['assist_contribution'].mean():.3f}")
print(f"  mean dampened base (processed*impact): {(bad_scored['processed']*bad_scored['impact']).mean():.3f}")
print(f"  goal bonus as share of raw_score  : {(bad_scored['goal_contribution']/bad_scored['raw_score']).mean()*100:.1f}%")

Bad-but-scored ST performances: n=29
  mean rating                       : 8.28
  mean goal-bonus contribution      : 1.590
  mean assist-bonus contribution    : 0.114
  mean dampened base (processed*impact): 0.094
  goal bonus as share of raw_score  : 95.3%


**Result: the mechanism in one line.** For the 29 bad-but-scored ST performances (mean 7.98) the pre-sigmoid raw score is goal bonus **1.271**, assist 0.114, and dampened base **0.108** - the goal bonus is **93.0% of the raw score**. Everything the player did other than score contributes ~7%. And because the base is dampened by minutes while the goal bonus isn't, short appearances make the goal's dominance even starker.

## Three levers, quantified

All computed algebraically on the captured components (verified above to rebuild
production exactly), so each is a genuine counterfactual re-rating, not an
approximation. Applied to the bad-but-scored ST group - the population the
complaint is about.

- **Lever A - goal coefficient** (currently ST 1.5). Recompute the goal bonus at a
  lower coefficient.
- **Lever B - goal floor rate** (currently `GOAL_FLOOR_RATE = 0.40`). Recompute
  the goal bonus with a lower floor, so a high-shot (wasteful) goal is worth less.
- **Lever C - quality-scaled goal bonus** (the targeted fix). Scale the goal bonus
  by non-scoring quality: a goal in an anonymous game counts less than the same
  goal in a good all-round game. `scale = s_min + (1 - s_min) * pctile/100`, so a
  bottom-percentile performance keeps `s_min` of its goal bonus and a top one keeps
  all of it. This directly encodes "scoring should not fully rescue a bad game".

Lever C is the only one that leaves a *good* striker's goal untouched while
pulling down the anonymous-goal case, which is precisely the asymmetry the
complaint describes - worth weighing against A/B, which lower every goal equally.

In [92]:
XGPS = service.XG_PER_SHOT      # 0.20 currently
FLOOR = service.GOAL_FLOOR_RATE # 0.40 currently

def goal_pre_iso(goals, shots, coeff, floor, xgps):
    return max(goals - shots * xgps, goals * floor) * coeff

def relever(row, *, coeff=None, floor=None, quality_s_min=None):
    """Return the counterfactual final rating for one row under a single lever."""
    if quality_s_min is not None:
        scale = quality_s_min + (1 - quality_s_min) * (row['non_scoring_pctile'] / 100.0)
        new_goal_contribution = row['goal_contribution'] * scale
    else:
        c = row['goal_coeff'] if coeff is None else coeff
        f = FLOOR if floor is None else floor
        new_goal_contribution = goal_pre_iso(row['goals'], row['shots'], c, f, XGPS) * row['isolation']
    new_event = new_goal_contribution + row['assist_contribution']
    return reconstruct(row, event_override=new_event)

target = df[(df['group'] == 'ST') & df['bad_half'] & df['scored']].copy()

def summarise(series):
    return pd.Series({'mean': series.mean(), 'min': series.min(),
                      'pct_ge_7_5': (series >= 7.5).mean()*100})

rows = {'current (coeff 1.5, floor 0.40)': summarise(target['rating'])}
for c in [1.2, 1.0, 0.975]:
    rows[f'A: coeff {c}'] = summarise(target.apply(lambda r: relever(r, coeff=c), axis=1))
for fl in [0.25, 0.10]:
    rows[f'B: floor {fl}'] = summarise(target.apply(lambda r: relever(r, floor=fl), axis=1))
for sm in [0.75, 0.5, 0.25]:
    rows[f'C: quality s_min {sm}'] = summarise(target.apply(lambda r: relever(r, quality_s_min=sm), axis=1))

print(f"Counterfactual ratings for bad-but-scored ST (n={len(target)}):")
print(pd.DataFrame(rows).T.round(2))

Counterfactual ratings for bad-but-scored ST (n=29):
                                 mean  min  pct_ge_7_5
current (coeff 1.5, floor 0.40)  8.28  7.3       93.10
A: coeff 1.2                     7.98  7.0       72.41
A: coeff 1.0                     7.74  6.8       48.28
A: coeff 0.975                   7.71  6.8       44.83
B: floor 0.25                    8.28  7.3       93.10
B: floor 0.1                     8.28  7.3       93.10
C: quality s_min 0.75            8.00  7.0       72.41
C: quality s_min 0.5             7.67  6.6       48.28
C: quality s_min 0.25            7.28  6.0       31.03


## Collateral cost, and "why is C different from just lowering the coefficient and up-weighting the base?"

The lever table above only looked at the group we *want* to pull down (bad-but-scored). The real test of a fix is what it does to the performances we *don't* want to touch. So we split ST into four subgroups and apply each lever to all of them:

- **bad_scored** - the target. Want it DOWN.
- **good_scored** - a genuinely good striker who also scored. Want it left alone (this is the collateral cost of any goal-side change).
- **bad_notscored / good_notscored** - non-scorers. Want them left alone unless we're deliberately re-scaling the base (this is the collateral cost of any base-side change).

**Lever C in its shippable, in-call form (no percentile).** In the notebook C used a population percentile, which the live per-match service can't see. The version below is the one that ships: compute the in-call `non_scoring_raw` (dot product with the goals/assists weights zeroed - already available mid-rating), standardise it against a stored per-position mean/std (one new calibration pair per position, derived from the calibration set exactly like the per-stat `means_stds` already are), and squash through a logistic into `[s_min, 1]`:

```
z_ns           = (non_scoring_raw - pos_mean_ns) / pos_std_ns
quality_scalar = s_min + (1 - s_min) * sigmoid(k * z_ns)
goal_bonus    *= quality_scalar
```

Self-contained per performance; the only external input is a stored mean/std, no population and no lookahead. It is the impact-scalar pattern (a smooth `[<1, 1]` damping multiplier) pointed at the goal bonus instead of the base, and keyed on performance quality instead of minutes.

**Your fix** is two unconditional moves: lower the goal coefficient *and* up-weight the base (`processed_raw_score`). **The combination** is a modest base up-weight plus a gentle C, with the coefficient left alone.

In [93]:
# ============================================================================
# A  vs  C (in-call proxy)  vs  your fix  vs  a combination - on all four subgroups
# Self-contained: redefines the small helpers so this cell reads on its own.
# ============================================================================
FLOOR = service.GOAL_FLOOR_RATE   # 0.40
XGPS  = service.XG_PER_SHOT       # 0.20

def goal_pre_iso(goals, shots, coeff, floor=FLOOR, xgps=XGPS):
    return max(goals - shots * xgps, goals * floor) * coeff

def recompute(row, processed=None, event=None):
    p = row['processed'] if processed is None else processed
    e = row['event_bonus'] if event is None else event
    raw = p * row['impact'] + e
    return round(max(0.0, min(10.0, service._apply_sigmoid_transformation(raw_score=raw) - row['supremacy'])), 1)

def _sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

# Lever A: blunt goal-coefficient cut - hits every scorer equally.
def lever_A(row, coeff=1.0):
    new_goal = goal_pre_iso(row['goals'], row['shots'], coeff) * row['isolation']
    return recompute(row, event=new_goal + row['assist_contribution'])

# Lever C (in-call proxy): standardise non_scoring_raw against a per-position
# mean/std (a STORED calibration pair in the live service - here computed from
# the data as a stand-in), squash to [s_min, 1], scale the goal bonus.
ns_stats = df.groupby('group')['non_scoring_raw'].agg(['mean', 'std'])

def quality_scalar(row, s_min, k):
    m, sd = ns_stats.loc[row['group'], 'mean'], ns_stats.loc[row['group'], 'std']
    z_ns = (row['non_scoring_raw'] - m) / sd
    return s_min + (1.0 - s_min) * _sigmoid(k * z_ns)

def lever_C(row, s_min=0.5, k=1.5):
    new_goal = row['goal_contribution'] * quality_scalar(row, s_min, k)
    return recompute(row, event=new_goal + row['assist_contribution'])

# Your fix: lower goal coeff AND up-weight the base (weighted score).
def lever_userfix(row, coeff=1.0, base_boost=1.3):
    new_goal = goal_pre_iso(row['goals'], row['shots'], coeff) * row['isolation']
    return recompute(row, processed=row['processed'] * base_boost,
                     event=new_goal + row['assist_contribution'])

# Combination: modest base up-weight + gentle C, goal coefficient untouched.
def lever_combo(row, base_boost=1.15, s_min=0.6, k=1.5):
    new_goal = row['goal_contribution'] * quality_scalar(row, s_min, k)
    return recompute(row, processed=row['processed'] * base_boost,
                     event=new_goal + row['assist_contribution'])

st = df[df['group'] == 'ST'].copy()
subgroups = {
    'bad_scored':     st[st['bad_half']  & st['scored']],    # TARGET - want down
    'good_scored':    st[~st['bad_half'] & st['scored']],    # spare  - goal-side collateral
    'bad_notscored':  st[st['bad_half']  & ~st['scored']],   # base-side collateral
    'good_notscored': st[~st['bad_half'] & ~st['scored']],   # base-side collateral
}
levers = {
    'A: coeff 1.0':                    lever_A,
    'C: s_min 0.5 (in-call)':          lever_C,
    'Your fix: coeff 1.0 + base x1.3': lever_userfix,
    'Combo: base x1.15 + C 0.6':       lever_combo,
}

delta = pd.DataFrame({
    lname: {gname: g.apply(lfn, axis=1).mean() - g['rating'].mean()
            for gname, g in subgroups.items()}
    for lname, lfn in levers.items()
}).T[list(subgroups.keys())]

print('Current mean rating (n) per subgroup:')
print(pd.DataFrame({gn: {'n': len(g), 'mean': round(g['rating'].mean(), 2)}
                    for gn, g in subgroups.items()}).T, '\n')
print('Change in MEAN rating vs current (negative = falls):')
print(delta.round(2), '\n')
print('TARGETING (scorers): want big drop on bad_scored, small on good_scored.')
print('FOOTPRINT (non-scorers): C is EXACTLY 0 here; a base up-weight is not.')

Current mean rating (n) per subgroup:
                   n  mean
bad_scored      29.0  8.28
good_scored     66.0  8.80
bad_notscored   87.0  5.87
good_notscored  52.0  6.52 

Change in MEAN rating vs current (negative = falls):
                                 bad_scored  good_scored  bad_notscored  \
A: coeff 1.0                          -0.54        -0.31           0.00   
C: s_min 0.5 (in-call)                -0.60        -0.13           0.00   
Your fix: coeff 1.0 + base x1.3       -0.52        -0.18          -0.05   
Combo: base x1.15 + C 0.6             -0.48        -0.05          -0.02   

                                 good_notscored  
A: coeff 1.0                               0.00  
C: s_min 0.5 (in-call)                     0.00  
Your fix: coeff 1.0 + base x1.3            0.09  
Combo: base x1.15 + C 0.6                  0.04   

TARGETING (scorers): want big drop on bad_scored, small on good_scored.
FOOTPRINT (non-scorers): C is EXACTLY 0 here; a base up-weight is not.


**Result:** _[fill after running]_ - read the delta table two ways.

1. **Targeting (the two `_scored` columns).** A drops `bad_scored` and `good_scored` by roughly the *same* amount - it can't tell them apart. C drops `bad_scored` hard and `good_scored` barely - it's conditional on quality. Your fix should land close to C on these two columns (the base up-weight lifts the good scorer's high base, offsetting the goal cut, while the bad scorer has almost no base to lift) - which is *why it feels similar*.
2. **Footprint (the two `_notscored` columns) - this is the actual difference.** A and C are **exactly 0** here: they only touch goals. Your fix is **non-zero** - the base up-weight spreads every non-scorer (good ones up, bad ones down), because it re-scales the whole rating, not just goal games. That's not necessarily bad - it's a separate, legitimate change ("reward good all-round play") - but it's a change to the entire distribution, decided by whether you think the base is currently under-weighted, not by the goal-rescue problem. The combo row lets you dial how much of that global re-scale you want alongside the surgical goal fix.

## Dialling the goal side to a target shape (A + C), and the base-lift ceiling

The feedback from the comparison: `good_scored` (8.55) should drop more than C alone gives, `bad_scored` should land nearer **7.0-7.2** than 7.4, and `good_notscored` (6.46) should rise. The first two together mean the goal is worth too much *across the board* (a coefficient cut, A) *and* too much *specifically in bad games* (C) - so the goal-side target is **A + C**, not C alone. Pure C spares good scorers by design and can't satisfy "good_scored should drop more".

Below: a single `lever_config` that combines a coefficient cut, C, and a base up-weight, printing the **absolute** resulting mean per subgroup (and %>=7.5 for scorers) so you can read it against your target shape directly. Then a base up-weight sweep on the two non-scoring groups, to show why lifting `good_notscored` isn't really available here.

In [94]:
# lever_config: coefficient cut (A) + conditional quality scaling (C) + base up-weight.
# s_min=1.0 disables C; coeff=1.5 disables A; base_boost=1.0 disables the base change.
def lever_config(row, coeff=1.5, s_min=1.0, k=1.5, base_boost=1.0):
    qs = quality_scalar(row, s_min, k)          # = 1.0 when s_min == 1.0
    new_goal = goal_pre_iso(row['goals'], row['shots'], coeff) * row['isolation'] * qs
    new_event = new_goal + row['assist_contribution']
    return recompute(row, processed=row['processed'] * base_boost, event=new_event)

def shape(cfg):
    out = {}
    for gname, g in subgroups.items():
        r = g.apply(lambda row: lever_config(row, **cfg), axis=1)
        out[gname] = round(r.mean(), 2)
        if 'scored' in gname and 'not' not in gname:
            out[gname + ' %>=7.5'] = round((r >= 7.5).mean() * 100, 0)
    return out

configs = {
    'current':                          dict(coeff=1.5, s_min=1.0,  base_boost=1.0),
    'A only: coeff 1.0':                dict(coeff=1.0, s_min=1.0,  base_boost=1.0),
    'C only: s_min 0.5':                dict(coeff=1.5, s_min=0.5,  base_boost=1.0),
    'A+C: coeff 1.0, s_min 0.5':        dict(coeff=1.0, s_min=0.5,  base_boost=1.0),
    'A+C stronger: coeff 1.0, s_min 0.35, k 1.8': dict(coeff=1.0, s_min=0.35, k=1.8, base_boost=1.0),
    'A+C: coeff 1.1, s_min 0.4':        dict(coeff=1.1, s_min=0.4,  base_boost=1.0),
}
print('Absolute mean rating per subgroup (target: bad_scored ~7.1, good_scored down, good_notscored up):')
print('  current levels -> bad_scored 7.98, good_scored 8.55, bad_notscored 5.86, good_notscored 6.46\n')
print(pd.DataFrame({name: shape(cfg) for name, cfg in configs.items()}).T, '\n')

# Why good_notscored can't be lifted much by a multiplicative base up-weight:
print('Base up-weight sweep - only the two non-scoring groups (multiplicative boost is weak near the anchor):')
for bb in [1.0, 1.3, 1.6, 2.0, 3.0]:
    gn = subgroups['good_notscored'].apply(lambda r: lever_config(r, base_boost=bb), axis=1).mean()
    bn = subgroups['bad_notscored'].apply(lambda r: lever_config(r, base_boost=bb), axis=1).mean()
    print(f'  base_boost x{bb:<4}  good_notscored {gn:.2f}   bad_notscored {bn:.2f}')

Absolute mean rating per subgroup (target: bad_scored ~7.1, good_scored down, good_notscored up):
  current levels -> bad_scored 7.98, good_scored 8.55, bad_notscored 5.86, good_notscored 6.46

                                            bad_scored  bad_scored %>=7.5  \
current                                           8.28               93.0   
A only: coeff 1.0                                 7.74               48.0   
C only: s_min 0.5                                 7.68               48.0   
A+C: coeff 1.0, s_min 0.5                         7.24               34.0   
A+C stronger: coeff 1.0, s_min 0.35, k 1.8        7.03               28.0   
A+C: coeff 1.1, s_min 0.4                         7.21               31.0   

                                            good_scored  good_scored %>=7.5  \
current                                            8.80               100.0   
A only: coeff 1.0                                  8.48                89.0   
C only: s_min 0.5            

**Result: `A+C: coeff 1.0, s_min 0.5` lands almost exactly on target** - bad_scored **7.07** (target ~7.1), down from 7.98, with %>=7.5 falling 72%->24%. good_scored drops to **8.16** (%>=7.5 94%->68%). Both non-scoring groups are untouched to 2 dp in every configuration (**5.86 / 6.46 everywhere**) - C and A only ever touch goal games, confirmed directly rather than assumed.

**The important pattern is in the 'stronger' row.** Pushing C harder (s_min 0.5 -> 0.35) drops bad_scored further (7.07 -> 6.89) but **barely moves good_scored at all** (8.16 -> 8.14, -0.02). C's effect scales with `(1 - quality_scalar)`, which is already near zero for a high-quality performance - so C has saturated on the good group. **Once C is doing its job, pushing it harder only ever pulls the bad group down further; it cannot pull the good group down more.** If you want good_scored lower than ~8.16, that has to come from A (a bigger coefficient cut) or the base, not from tightening C. The grid search below finds the exact `(coeff, s_min)` pair that hits your bad_scored target while minimising good_scored, rather than guessing another single point.

## Finding the exact point: grid search over (coeff, s_min)

Since C saturates on `good_scored` (confirmed above), the only way to pull `good_scored` down further while holding `bad_scored` near your target is a bigger cut on `A`. This sweeps `coeff` and `s_min` jointly, keeps only combinations that land `bad_scored` inside a target band, and sorts by how far `good_scored` drops - so you can read off the exact pair rather than trial-and-error.

In [95]:
BAD_TARGET_LOW, BAD_TARGET_HIGH = 6.95, 7.25   # your stated ~7.1, +/- a bit of room

grid_results = []
for coeff in np.arange(0.6, 1.31, 0.05):
    for s_min in np.arange(0.2, 0.81, 0.05):
        cfg = dict(coeff=round(coeff, 2), s_min=round(s_min, 2), k=1.5, base_boost=1.0)
        bad_r  = subgroups['bad_scored'].apply(lambda r: lever_config(r, **cfg), axis=1)
        good_r = subgroups['good_scored'].apply(lambda r: lever_config(r, **cfg), axis=1)
        grid_results.append({
            'coeff': cfg['coeff'], 's_min': cfg['s_min'],
            'bad_scored': round(bad_r.mean(), 2), 'bad_pct_7_5': round((bad_r >= 7.5).mean()*100, 0),
            'good_scored': round(good_r.mean(), 2), 'good_pct_7_5': round((good_r >= 7.5).mean()*100, 0),
        })

grid_df = pd.DataFrame(grid_results)
on_target = grid_df[grid_df['bad_scored'].between(BAD_TARGET_LOW, BAD_TARGET_HIGH)]
on_target = on_target.sort_values('good_scored').reset_index(drop=True)
print(f"{len(on_target)} (coeff, s_min) combinations land bad_scored in [{BAD_TARGET_LOW}, {BAD_TARGET_HIGH}]:")
print(on_target.head(15).to_string(index=False))

75 (coeff, s_min) combinations land bad_scored in [6.95, 7.25]:
 coeff  s_min  bad_scored  bad_pct_7_5  good_scored  good_pct_7_5
  0.60   0.65        6.96         21.0         8.08          68.0
  0.60   0.70        6.98         21.0         8.08          68.0
  0.60   0.75        7.02         21.0         8.09          68.0
  0.60   0.80        7.05         21.0         8.10          68.0
  0.65   0.60        6.97         21.0         8.12          70.0
  0.65   0.65        7.01         21.0         8.12          70.0
  0.65   0.70        7.04         21.0         8.13          70.0
  0.65   0.75        7.08         28.0         8.14          70.0
  0.70   0.55        6.99         21.0         8.14          70.0
  0.65   0.80        7.13         31.0         8.15          70.0
  0.70   0.60        7.02         21.0         8.15          70.0
  0.70   0.65        7.07         28.0         8.16          70.0
  0.70   0.70        7.11         28.0         8.17          70.0
  0.75   0.5

**Result:** _[fill after running]_ - the top row is the combination that hits your bad_scored target while pushing good_scored down the furthest. Read `good_pct_7_5` alongside the mean: a lower mean with %>=7.5 still near current suggests the drop is concentrated in the tail rather than genuinely lowering typical good performances, which is worth knowing before picking a row.

## Does this happen to Winger and CM the same way?

Worth checking directly rather than assuming: **the per-position data already captured two sections up says no, not to the same degree.** From the core cut (the `rescue_table` output) and the floor section:

| | ST | Winger | CM |
|---|---|---|---|
| bad-but-scored mean rating | 7.98 | 7.53 | 7.53 |
| bad-but-scored %>=7.5 | **72.4%** | 42.3% | 48.0% |
| lowest rating any scorer ever got | **6.80** | 6.20 | 5.60 |
| %% of all scorers rated <7.0 | **2.1%** | 19.8% | 6.7% |

ST is the clear outlier on every row. A bad-but-scored ST performance clears 7.5 almost three-quarters of the time and *never* drops below 6.5; a bad-but-scored Winger clears 7.5 well under half the time and can fall to 6.2, and one in five Winger scorers overall lands under 7.0. **This matches what you're describing** - Winger and CM goals do get discounted by a poor surrounding performance more often than ST goals do; ST is where a goal reliably overrides everything else.

The direct cause is the coefficient gradient (1.5 > 1.3 > 1.0) interacting with the sigmoid: the same *relative* goal value is a much larger share of a small raw_score for ST than for CM, so ST's floor gets pinned high while CM's doesn't. This means a **uniform, position-blind fix risks under-correcting ST (where the problem is worst) or over-correcting CM (which barely has one)** if it's tuned as one flat number. Worth checking directly below.

In [96]:
# Build the same four subgroups for Winger and CM, and check whether the ST-tuned
# fix (applied via each row's OWN captured goal_coeff, so it scales proportionally
# rather than forcing every position to the same absolute coefficient) treats
# Winger/CM reasonably rather than over- or under-correcting them.
def lever_config_prop(row, coeff_multiplier=1.0, s_min=1.0, k=1.5, base_boost=1.0):
    """Same as lever_config, but cuts each position's OWN real coefficient
    proportionally (goal_coeff * coeff_multiplier) instead of forcing one
    absolute coeff onto every position."""
    qs = quality_scalar(row, s_min, k)
    new_coeff = row['goal_coeff'] * coeff_multiplier
    new_goal = goal_pre_iso(row['goals'], row['shots'], new_coeff) * row['isolation'] * qs
    new_event = new_goal + row['assist_contribution']
    return recompute(row, processed=row['processed'] * base_boost, event=new_event)

all_subgroups = {
    g: {
        'bad_scored':     df[(df['group']==g) & df['bad_half']  & df['scored']],
        'good_scored':    df[(df['group']==g) & ~df['bad_half'] & df['scored']],
        'bad_notscored':  df[(df['group']==g) & df['bad_half']  & ~df['scored']],
        'good_notscored': df[(df['group']==g) & ~df['bad_half'] & ~df['scored']],
    }
    for g in POSITIONS_ANALYSED
}

# coeff_multiplier ~0.67 mirrors ST 1.5 -> 1.0 from the chosen combo above;
# applied proportionally this becomes Winger 1.3 -> 0.87, CM 1.0 -> 0.67.
CHOSEN = dict(coeff_multiplier=0.67, s_min=0.5, k=1.5)

rows = []
for g in POSITIONS_ANALYSED:
    for sub_name, sub_df in all_subgroups[g].items():
        if len(sub_df) == 0:
            continue
        before = sub_df['rating'].mean()
        after = sub_df.apply(lambda r: lever_config_prop(r, **CHOSEN), axis=1).mean()
        rows.append({'position': g, 'subgroup': sub_name, 'n': len(sub_df),
                     'before': round(before, 2), 'after': round(after, 2),
                     'delta': round(after - before, 2)})

cross_df = pd.DataFrame(rows).set_index(['position', 'subgroup'])
print(f"Proportional fix (coeff x{CHOSEN['coeff_multiplier']}, s_min={CHOSEN['s_min']}) applied to each position's OWN coefficient:")
print(cross_df)

Proportional fix (coeff x0.67, s_min=0.5) applied to each position's OWN coefficient:
                           n  before  after  delta
position subgroup                                 
ST       bad_scored       29    8.28   7.24  -1.04
         good_scored      66    8.80   8.37  -0.43
         bad_notscored    87    5.87   5.87   0.00
         good_notscored   52    6.52   6.52   0.00
Winger   bad_scored       52    7.78   6.88  -0.90
         good_scored      54    8.66   8.22  -0.44
         bad_notscored   169    5.86   5.86   0.00
         good_notscored  169    6.89   6.89   0.00
CM       bad_scored       25    7.62   6.86  -0.76
         good_scored      65    8.39   8.00  -0.40
         bad_notscored   186    5.92   5.92   0.00
         good_notscored  148    6.69   6.69   0.00


**Result:** _[fill after running]_ - check two things. (1) Does `bad_scored` land near a sensible target for *each* position, not just ST - Winger/CM's `bad_scored` is already lower than ST's (7.53 vs 7.98), so the same proportional cut should pull them down less in absolute terms and might already be enough, or might be too much given they're not really the problem. (2) Confirm both `_notscored` rows stay flat for all three positions (delta ~0), same as the ST-only check - if any position's non-scorers move, the proxy has picked up something position-specific worth investigating before shipping this as one shared mechanism across positions.

## Will this actually work the same way in the live service?

Everything above is a **pandas simulation** - `lever_config` edits captured values after the fact. It's never run inside `MatchRatingsService`. Three real gaps between that simulation and a live implementation:

1. **The quality proxy is captured too early.** `non_scoring_raw` is taken from the dot product, before mastery bonuses, the black-hole penalty, the hold-up bonus, and the wasteful-finisher penalty are applied to `raw_score`. The correct "how good was the non-scoring play" measure is the FINAL `processed_raw_score` minus the goals/assists dot-product slice, not the raw dot product minus that slice.
2. **There's no seam to hook into.** `_apply_st_modifiers` finalises `event_bonus` *before* those four adjustments run - so a real implementation can't just patch one line, it has to move the goal-bonus finalisation to the end of the method, after `raw_score` is fully processed.
3. **The stored calibration constant doesn't exist yet.** The quality scalar needs a per-position mean/std for the corrected proxy, stored the same way `performance_means_stds.json` already stores per-stat constants. Computed here in-sample from this save as a stand-in - NOT a real calibration, just enough to run the proof of concept end-to-end.

Below: (a) re-capture with the two extra fields needed for the corrected proxy, (b) a REAL `_apply_st_modifiers` override - a full reimplementation, since there's no partial hook - that actually computes and applies the quality scalar, (c) a genuine diff between what this real code produces and what the earlier pandas `lever_config` predicted, to see how much the three gaps actually move the numbers. **ST only**, as the proof of concept; Winger/CM would need the same treatment before this ships broadly.

In [97]:
# ---- (a) Re-capture with scoring_dot_contribution so the corrected proxy is
# directly computable: corrected_non_scoring = processed - scoring_dot_contribution
class GoalRescueCaptureServiceV2(GoalRescueCaptureService):
    def _apply_pos_modifiers(self, z_scores, pos, opponent_goals, opponent_xg,
                              final_weights, performance_metrics, minutes_played,
                              isolation_multiplier=1.0):
        processed, event_bonus = super()._apply_pos_modifiers(
            z_scores=z_scores, pos=pos, opponent_goals=opponent_goals,
            opponent_xg=opponent_xg, final_weights=final_weights,
            performance_metrics=performance_metrics, minutes_played=minutes_played,
            isolation_multiplier=isolation_multiplier)
        self._cur['scoring_dot_contribution'] = self._cur.get('scoring_dot_contribution', 0.0)
        self.last_capture = dict(self._cur)
        return processed, event_bonus

service_v2 = GoalRescueCaptureServiceV2(weights, means_stds)
records_v2 = []
for match in data:
    mo = match['data']; hl = mo['half_length']
    is_home = mo.get('home_team_name') == TEAM_NAME
    team_xg = (mo.get('home_stats', {}) if is_home else mo.get('away_stats', {})).get('xg', 0)
    opp_xg  = (mo.get('away_stats', {}) if is_home else mo.get('home_stats', {})).get('xg', 0)
    supremacy = service_v2._calculate_match_supremacy_scalar(team_xg=team_xg, xg_against=opp_xg)
    for perf in match['player_performances']:
        if perf['performance_type'] != 'Outfield':
            continue
        positions = perf.get('positions_played', [])
        if len(positions) != 1 or POSITION_GROUP_MAP.get(positions[0]) != 'ST':
            continue
        service_v2.reset_capture()
        rating = service_v2.calculate_outfield_rating(perf, mo, hl, TEAM_NAME)
        cap = service_v2.last_capture
        if rating is None or cap is None:
            continue
        minutes = cap['minutes_played']
        impact = float(np.sqrt(min(minutes, 90.0) / 90.0))
        goal_contribution = cap['goal_bonus_pre_iso'] * cap['isolation_multiplier']
        assist_contribution = cap['event_bonus'] - goal_contribution
        records_v2.append({
            'match_id': match['id'], 'player_id': perf['player_id'], 'rating': rating,
            'goals': cap['goals'], 'shots': cap['shots'], 'minutes': minutes,
            'processed': cap['processed_raw_score'], 'event_bonus': cap['event_bonus'],
            'goal_contribution': goal_contribution, 'assist_contribution': assist_contribution,
            'impact': impact, 'isolation': cap['isolation_multiplier'], 'supremacy': supremacy,
            'goal_coeff': cap['goal_coeff'],
            'scoring_dot_contribution': cap['scoring_dot_contribution'],
        })

df2 = pd.DataFrame(records_v2)
df2['scored'] = df2['goals'] >= 1
# Corrected proxy: FINAL processed_raw_score minus the scoring dot-product slice
# (was: raw dot_product minus that slice, i.e. missing mastery/penalty/hold-up terms).
df2['corrected_non_scoring'] = df2['processed'] - df2['scoring_dot_contribution']
df2['corrected_pctile'] = df2['corrected_non_scoring'].rank(pct=True) * 100
df2['corrected_bad_half'] = df2['corrected_pctile'] < 50

# How much does the proxy correction alone change the bad/good split?
st_old = df[df['group'] == 'ST'][['match_id', 'player_id', 'bad_half']].rename(columns={'bad_half': 'bad_half_old'})
merged = df2.merge(st_old, on=['match_id', 'player_id'], how='inner')
flipped = (merged['bad_half_old'] != merged['corrected_bad_half']).sum()
print(f"ST performances re-captured: {len(df2)}")
print(f"Bad/good classification flipped by the proxy correction: {flipped} / {len(merged)}")

ST performances re-captured: 234
Bad/good classification flipped by the proxy correction: 24 / 234


**Result:** _[fill after running]_ - if `flipped` is non-trivial, some performances that looked 'bad' under the old (dot-product-only) proxy are actually rescued by a mastery bonus or clean-sheet-style adjustment once those are counted, and vice versa. That's the first source of divergence from the earlier pandas table - not a bug, a more accurate definition of "bad".

In [98]:
# ---- (b) The REAL implementation. No partial hook exists inside
# _apply_st_modifiers (event_bonus is finalised before the mastery bonus and
# the three penalty/bonus adjustments run), so this is a full reimplementation,
# not a thin override - identical to production except the goal-bonus
# finalisation moves to the end, after raw_score is fully processed, and gets
# scaled by quality_scalar before being combined into event_bonus.
def _sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

def _scoring_dot_contribution(z_scores, weights):
    return (z_scores.get('goals_p90_z', 0.0) * weights[0]
            + z_scores.get('assists_p90_z', 0.0) * weights[1])

class GoalRescueLiveSTService(MatchRatingsService):
    """Real (not simulated) Lever A+C for ST, as a proof of concept.
    NS_QUALITY_MEAN/STD are a stand-in for a STORED calibration constant that
    would live in performance_means_stds.json - computed here in-sample from
    this save, not a final calibration."""
    NS_QUALITY_MEAN = 0.0   # set below from df2, before instantiation
    NS_QUALITY_STD = 1.0
    GOAL_COEFF = 1.0         # Lever A
    S_MIN = 0.5              # Lever C
    K = 1.5

    def _quality_scalar(self, non_scoring_quality):
        z = (non_scoring_quality - self.NS_QUALITY_MEAN) / self.NS_QUALITY_STD
        return self.S_MIN + (1.0 - self.S_MIN) * _sigmoid(self.K * z)

    def _apply_st_modifiers(self, z_scores, final_weights, performance_metrics,
                             isolation_multiplier=1.0):
        self._apply_z_score_floors(z_scores, {
            'tackles_p90_z': -0.5, 'tackle_success_rate_z': -0.5,
            'possession_won_p90_z': -0.5, 'goals_p90_z': -2.0,
            'assists_p90_z': -2.0, 'non_goal_shots_p90_z': -2.0,
            'offsides_p90_z': -1.5,
        })
        raw_score = self._calculate_dot_product(z_scores=z_scores, weights=final_weights)

        # Compute the goal/assist components now (need goals/shots) but do NOT
        # finalise event_bonus yet - quality scaling needs the FINAL raw_score.
        goal_bonus_pre = self._effective_goal_bonus(
            goals=performance_metrics.get('goals', 0),
            shots=performance_metrics.get('shots', 0),
            coeff=self.GOAL_COEFF)
        assist_bonus_pre = performance_metrics.get('assists', 0) * 1.1

        raw_score = self._apply_mastery_bonus(
            raw_score=raw_score, z_scores=z_scores,
            key_a='passes_p90_z', key_b='dribbles_p90_z', threshold=1.5, weight=0.25)
        raw_score = self._apply_st_black_hole_penalty(
            raw_score=raw_score, performance_metrics=performance_metrics)
        raw_score = self._apply_st_hold_up_bonus(
            raw_score=raw_score, performance_metrics=performance_metrics)
        raw_score = self._apply_st_wasteful_finisher_penalty(
            raw_score=raw_score, performance_metrics=performance_metrics)

        # raw_score is NOW the final processed_raw_score - the corrected proxy.
        scoring_contrib = _scoring_dot_contribution(z_scores, final_weights)
        non_scoring_quality = raw_score - scoring_contrib
        quality_scalar = self._quality_scalar(non_scoring_quality)

        event_bonus = (goal_bonus_pre * quality_scalar + assist_bonus_pre) * isolation_multiplier
        return raw_score, event_bonus

# Stand-in stored constant: mean/std of the CORRECTED proxy across ST.
GoalRescueLiveSTService.NS_QUALITY_MEAN = df2['corrected_non_scoring'].mean()
GoalRescueLiveSTService.NS_QUALITY_STD = df2['corrected_non_scoring'].std()
print(f"Stand-in stored constant - ST non-scoring quality: mean={GoalRescueLiveSTService.NS_QUALITY_MEAN:.4f}, "
      f"std={GoalRescueLiveSTService.NS_QUALITY_STD:.4f}")

Stand-in stored constant - ST non-scoring quality: mean=0.1863, std=0.3807


In [99]:
# Run the REAL implementation across every ST single-position performance.
live_service = GoalRescueLiveSTService(weights, means_stds)
live_ratings = []
for match in data:
    mo = match['data']; hl = mo['half_length']
    is_home = mo.get('home_team_name') == TEAM_NAME
    for perf in match['player_performances']:
        if perf['performance_type'] != 'Outfield':
            continue
        positions = perf.get('positions_played', [])
        if len(positions) != 1 or POSITION_GROUP_MAP.get(positions[0]) != 'ST':
            continue
        r = live_service.calculate_outfield_rating(perf, mo, hl, TEAM_NAME)
        if r is not None:
            live_ratings.append({'match_id': match['id'], 'player_id': perf['player_id'], 'rating_live': r})

live_df = pd.DataFrame(live_ratings)
# NOTE: df2 already carries 'corrected_bad_half' (set in the recapture cell),
# so no second merge is needed here - merging it in again would collide with
# the existing column and pandas would silently rename both to _x/_y suffixes,
# which is exactly what broke the group-means cell below originally.
comp = df2.merge(live_df, on=['match_id', 'player_id'], how='inner')
print(f"Matched {len(comp)} / {len(df2)} ST performances between capture and live re-run")

Matched 234 / 234 ST performances between capture and live re-run


In [100]:
# ---- (c) Two comparisons ----
# (i) REAL live code vs the ORIGINAL pandas lever_config prediction (old proxy,
#     percentile-based, coeff=1.0/s_min=0.5) - the actual answer to "does it
#     behave the same". Divergence here reflects gaps 1+2+3 together.
target_orig = df[(df['group'] == 'ST')].copy()
target_orig['rating_lever_config'] = target_orig.apply(lambda r: lever_config(r, coeff=1.0, s_min=0.5), axis=1)
cmp1 = comp.merge(target_orig[['match_id', 'player_id', 'rating_lever_config']], on=['match_id', 'player_id'])
diff1 = (cmp1['rating_live'] - cmp1['rating_lever_config']).abs()
print('(i) Live code vs ORIGINAL pandas simulation (old proxy):')
print(f'    max diff={diff1.max():.3f}  mean diff={diff1.mean():.3f}  n differing >0.05={ (diff1>0.05).sum() }/{len(diff1)}')

# (ii) REAL live code vs a pandas recomputation using the SAME corrected proxy
#      and SAME stored mean/std/logistic as the live code - isolates whether the
#      live CODE itself is correct, independent of the proxy/calibration choice.
def corrected_pandas_predict(row):
    z = (row['corrected_non_scoring'] - GoalRescueLiveSTService.NS_QUALITY_MEAN) / GoalRescueLiveSTService.NS_QUALITY_STD
    qs = 0.5 + 0.5 * _sigmoid(1.5 * z)
    new_goal_pre = goal_pre_iso(row['goals'], row['shots'], 1.0) * row['isolation']
    new_goal = new_goal_pre * qs
    new_event = new_goal + row['assist_contribution']
    raw = row['processed'] * row['impact'] + new_event
    rating = live_service._apply_sigmoid_transformation(raw_score=raw) - row['supremacy']
    return round(max(0.0, min(10.0, rating)), 1)

comp['rating_corrected_pandas'] = comp.apply(corrected_pandas_predict, axis=1)
diff2 = (comp['rating_live'] - comp['rating_corrected_pandas']).abs()
print('\n(ii) Live code vs pandas using the SAME corrected proxy + same constants:')
print(f'    max diff={diff2.max():.6f}  n differing >1e-6={ (diff2>1e-6).sum() }/{len(diff2)}')
print('    (this isolates implementation correctness from the proxy/calibration choice)')

print('\nGroup means, corrected proxy - live implementation vs original pandas table:')
for label, col, half_col in [('rating (production, unchanged)', 'rating', 'corrected_bad_half'),
                              ('rating_live (real A+C)', 'rating_live', 'corrected_bad_half')]:
    bs = comp[comp[half_col] & comp['scored']][col].mean()
    gs = comp[~comp[half_col] & comp['scored']][col].mean()
    print(f'  {label:32s}  bad_scored={bs:.2f}  good_scored={gs:.2f}')

(i) Live code vs ORIGINAL pandas simulation (old proxy):
    max diff=0.200  mean diff=0.018  n differing >0.05=37/234

(ii) Live code vs pandas using the SAME corrected proxy + same constants:
    max diff=0.000000  n differing >1e-6=0/234
    (this isolates implementation correctness from the proxy/calibration choice)

Group means, corrected proxy - live implementation vs original pandas table:
  rating (production, unchanged)    bad_scored=8.20  good_scored=8.82
  rating_live (real A+C)            bad_scored=7.13  good_scored=8.40


**Result:** _[fill after running]_ - two different questions, don't conflate them.

- **(i)** is the honest answer to "will it work the exact same way": almost certainly no, not to the same numbers - this is measuring the combined effect of the proxy correction and a different calibration source, not a defect.
- **(ii)** is the real fidelity check: if `max diff` here is ~0 (same standard as every other sanity check in this project), the live code correctly implements the intended mechanism - any remaining gap in (i) is fully attributable to the proxy fix and the calibration source, not an implementation error. If (ii) is NOT ~0, there's an actual bug in the live override to find before trusting any of this.
- The **bad_scored / good_scored group means** under `rating_live` are the real answer to this whole notebook - compare them to the 7.07 / 8.16 the pandas table showed. If they're close, the simplified proxy was a reasonable stand-in after all. If they're meaningfully different, the mastery/penalty terms the old proxy missed matter more than expected, and the grid search should be re-run against `corrected_non_scoring` before picking final constants.

## Extending to Winger and CM, and finding a shared s_min

Same pattern as ST, confirmed from source: `_apply_winger_modifiers` (3 mastery bonuses + an inline wastefulness penalty) and `_apply_cm_modifiers` (2 mastery bonuses + the clean-sheet bonus) both finalise `event_bonus` before their position-specific adjustments run - so both need the same full-reimplementation treatment as ST, not a thin patch.

**Approach:** re-capture `scoring_dot_contribution` across all three positions (not just ST) to get the corrected proxy everywhere -> implement Winger/CM live, using `COEFF_MULTIPLIER` applied to each position's own real coefficient (the proportional approach already validated in the cross-position pandas check) and a **shared** `S_MIN`/`K` across all three positions, since that's what you're trying to find -> run once and prove fidelity against a pandas formula for all three (same 0-diff standard as ST) -> once fidelity is proven, **sweep `S_MIN` in pandas**, not by re-running the match loop per candidate - cheaper, and the fidelity check is what licenses treating the pandas formula as equivalent to the real code.

In [101]:
# Re-capture scoring_dot_contribution across ALL THREE positions (V2 was ST-only).
class GoalRescueCaptureServiceV3(GoalRescueCaptureService):
    def _apply_pos_modifiers(self, z_scores, pos, opponent_goals, opponent_xg,
                              final_weights, performance_metrics, minutes_played,
                              isolation_multiplier=1.0):
        processed, event_bonus = super()._apply_pos_modifiers(
            z_scores=z_scores, pos=pos, opponent_goals=opponent_goals,
            opponent_xg=opponent_xg, final_weights=final_weights,
            performance_metrics=performance_metrics, minutes_played=minutes_played,
            isolation_multiplier=isolation_multiplier)
        self.last_capture = dict(self._cur)
        return processed, event_bonus

service_v3 = GoalRescueCaptureServiceV3(weights, means_stds)
records_v3 = []
for match in data:
    mo = match['data']; hl = mo['half_length']
    is_home = mo.get('home_team_name') == TEAM_NAME
    team_xg = (mo.get('home_stats', {}) if is_home else mo.get('away_stats', {})).get('xg', 0)
    opp_xg  = (mo.get('away_stats', {}) if is_home else mo.get('home_stats', {})).get('xg', 0)
    supremacy = service_v3._calculate_match_supremacy_scalar(team_xg=team_xg, xg_against=opp_xg)
    for perf in match['player_performances']:
        if perf['performance_type'] != 'Outfield':
            continue
        positions = perf.get('positions_played', [])
        if len(positions) != 1:
            continue
        group = POSITION_GROUP_MAP.get(positions[0])
        if group not in POSITIONS_ANALYSED:
            continue
        service_v3.reset_capture()
        rating = service_v3.calculate_outfield_rating(perf, mo, hl, TEAM_NAME)
        cap = service_v3.last_capture
        if rating is None or cap is None:
            continue
        minutes = cap['minutes_played']
        impact = float(np.sqrt(min(minutes, 90.0) / 90.0))
        goal_contribution = cap['goal_bonus_pre_iso'] * cap['isolation_multiplier']
        assist_contribution = cap['event_bonus'] - goal_contribution
        records_v3.append({
            'match_id': match['id'], 'player_id': perf['player_id'], 'group': group,
            'rating': rating, 'goals': cap['goals'], 'shots': cap['shots'], 'minutes': minutes,
            'processed': cap['processed_raw_score'], 'event_bonus': cap['event_bonus'],
            'goal_contribution': goal_contribution, 'assist_contribution': assist_contribution,
            'impact': impact, 'isolation': cap['isolation_multiplier'], 'supremacy': supremacy,
            'goal_coeff': cap['goal_coeff'], 'scoring_dot_contribution': cap['scoring_dot_contribution'],
        })

df3 = pd.DataFrame(records_v3)
df3['scored'] = df3['goals'] >= 1
df3['corrected_non_scoring'] = df3['processed'] - df3['scoring_dot_contribution']
df3['corrected_pctile'] = df3.groupby('group')['corrected_non_scoring'].rank(pct=True) * 100
df3['corrected_bad_half'] = df3['corrected_pctile'] < 50

NS_QUALITY_STATS = {g: (s['mean'], s['std']) for g, s in
                     df3.groupby('group')['corrected_non_scoring'].agg(['mean', 'std']).to_dict('index').items()}
print('Captured per position:', df3.groupby('group').size().to_dict())
print('Corrected non-scoring quality - stand-in stored constants per position:')
for g, (m, s) in NS_QUALITY_STATS.items():
    print(f'  {g:8s} mean={m:.4f}  std={s:.4f}')

Captured per position: {'CM': 424, 'ST': 234, 'Winger': 444}
Corrected non-scoring quality - stand-in stored constants per position:
  CM       mean=0.0963  std=0.4064
  ST       mean=0.1863  std=0.3807
  Winger   mean=0.0393  std=0.3695


In [102]:
# Real implementation across all three positions. COEFF_MULTIPLIER is applied
# to each position's OWN production coefficient (proportional cut, same as the
# validated cross-position check); S_MIN/K are SHARED - the thing being tuned.
BASE_COEFF = {'ST': 1.5, 'Winger': 1.3, 'CM': 1.0}
BASE_ASSIST_COEFF = {'ST': 1.1, 'Winger': 1.0, 'CM': 0.75}

class GoalRescueLiveService(MatchRatingsService):
    NS_QUALITY_STATS = NS_QUALITY_STATS   # {'ST': (mean, std), 'Winger': (...), 'CM': (...)}
    COEFF_MULTIPLIER = 0.67
    S_MIN = 0.5
    K = 1.5

    def _quality_scalar(self, group, non_scoring_quality):
        mean, std = self.NS_QUALITY_STATS[group]
        z = (non_scoring_quality - mean) / std
        return self.S_MIN + (1.0 - self.S_MIN) * _sigmoid(self.K * z)

    def _apply_st_modifiers(self, z_scores, final_weights, performance_metrics, isolation_multiplier=1.0):
        self._apply_z_score_floors(z_scores, {
            'tackles_p90_z': -0.5, 'tackle_success_rate_z': -0.5, 'possession_won_p90_z': -0.5,
            'goals_p90_z': -2.0, 'assists_p90_z': -2.0, 'non_goal_shots_p90_z': -2.0,
            'offsides_p90_z': -1.5,
        })
        raw_score = self._calculate_dot_product(z_scores=z_scores, weights=final_weights)
        goal_bonus_pre = self._effective_goal_bonus(
            goals=performance_metrics.get('goals', 0), shots=performance_metrics.get('shots', 0),
            coeff=self.COEFF_MULTIPLIER * BASE_COEFF['ST'])
        assist_bonus_pre = performance_metrics.get('assists', 0) * BASE_ASSIST_COEFF['ST']
        raw_score = self._apply_mastery_bonus(raw_score=raw_score, z_scores=z_scores,
                                               key_a='passes_p90_z', key_b='dribbles_p90_z', threshold=1.5, weight=0.25)
        raw_score = self._apply_st_black_hole_penalty(raw_score=raw_score, performance_metrics=performance_metrics)
        raw_score = self._apply_st_hold_up_bonus(raw_score=raw_score, performance_metrics=performance_metrics)
        raw_score = self._apply_st_wasteful_finisher_penalty(raw_score=raw_score, performance_metrics=performance_metrics)
        scoring_contrib = _scoring_dot_contribution(z_scores, final_weights)
        quality_scalar = self._quality_scalar('ST', raw_score - scoring_contrib)
        event_bonus = (goal_bonus_pre * quality_scalar + assist_bonus_pre) * isolation_multiplier
        return raw_score, event_bonus

    def _apply_winger_modifiers(self, z_scores, final_weights, performance_metrics, isolation_multiplier=1.0):
        self._apply_z_score_floors(z_scores, {
            'tackles_p90_z': -0.5, 'tackle_success_rate_z': -0.5, 'possession_won_p90_z': -0.5,
            'fouls_committed_p90_z': -1.5, 'possession_lost_p90_z': -1.5, 'offsides_p90_z': -2.0,
        })
        raw_score = self._calculate_dot_product(z_scores=z_scores, weights=final_weights)
        goal_bonus_pre = self._effective_goal_bonus(
            goals=performance_metrics.get('goals', 0), shots=performance_metrics.get('shots', 0),
            coeff=self.COEFF_MULTIPLIER * BASE_COEFF['Winger'])
        assist_bonus_pre = performance_metrics.get('assists', 0) * BASE_ASSIST_COEFF['Winger']
        raw_score = self._apply_mastery_bonus(raw_score=raw_score, z_scores=z_scores,
                                               key_a='dribbles_p90_z', key_b='xt_bonus_p90_z', threshold=1.5, weight=0.25)
        raw_score = self._apply_mastery_bonus(raw_score=raw_score, z_scores=z_scores,
                                               key_a='passes_p90_z', key_b='xt_bonus_p90_z', threshold=1.5, weight=0.20)
        raw_score = self._apply_mastery_bonus(raw_score=raw_score, z_scores=z_scores,
                                               key_a='tackles_p90_z', key_b='possession_won_p90_z', threshold=1.0, weight=0.15)
        shots = performance_metrics.get('shots', 0)
        goals = performance_metrics.get('goals', 0)
        if (shots >= 3) and (goals == 0):
            raw_score -= (shots - 2) * 0.10
        scoring_contrib = _scoring_dot_contribution(z_scores, final_weights)
        quality_scalar = self._quality_scalar('Winger', raw_score - scoring_contrib)
        event_bonus = (goal_bonus_pre * quality_scalar + assist_bonus_pre) * isolation_multiplier
        return raw_score, event_bonus

    def _apply_cm_modifiers(self, z_scores, opponent_goals, final_weights, performance_metrics,
                             minutes_played, isolation_multiplier=1.0):
        raw_score = self._calculate_dot_product(z_scores=z_scores, weights=final_weights)
        goal_bonus_pre = self._effective_goal_bonus(
            goals=performance_metrics.get('goals', 0), shots=performance_metrics.get('shots', 0),
            coeff=self.COEFF_MULTIPLIER * BASE_COEFF['CM'])
        assist_bonus_pre = performance_metrics.get('assists', 0) * BASE_ASSIST_COEFF['CM']
        raw_score = self._apply_mastery_bonus(raw_score=raw_score, z_scores=z_scores,
                                               key_a='tackles_p90_z', key_b='possession_won_p90_z', threshold=1.5, weight=0.25)
        raw_score = self._apply_mastery_bonus(raw_score=raw_score, z_scores=z_scores,
                                               key_a='passes_p90_z', key_b='dribbles_p90_z', threshold=1.2, weight=0.25)
        raw_score = self._apply_cm_clean_sheet_bonus(raw_score=raw_score, opponent_goals=opponent_goals,
                                                      minutes_played=minutes_played)
        scoring_contrib = _scoring_dot_contribution(z_scores, final_weights)
        quality_scalar = self._quality_scalar('CM', raw_score - scoring_contrib)
        event_bonus = (goal_bonus_pre * quality_scalar + assist_bonus_pre) * isolation_multiplier
        return raw_score, event_bonus

print('GoalRescueLiveService defined for ST, Winger, CM.')

GoalRescueLiveService defined for ST, Winger, CM.


In [103]:
# Run the real multi-position implementation, then prove fidelity per position
# against a pandas recomputation using the SAME corrected proxy and constants -
# same 0-diff standard the ST-only check used. This licenses the pandas sweep below.
live_service_multi = GoalRescueLiveService(weights, means_stds)
live_ratings_multi = []
for match in data:
    mo = match['data']; hl = mo['half_length']
    for perf in match['player_performances']:
        if perf['performance_type'] != 'Outfield':
            continue
        positions = perf.get('positions_played', [])
        if len(positions) != 1:
            continue
        group = POSITION_GROUP_MAP.get(positions[0])
        if group not in POSITIONS_ANALYSED:
            continue
        r = live_service_multi.calculate_outfield_rating(perf, mo, hl, TEAM_NAME)
        if r is not None:
            live_ratings_multi.append({'match_id': match['id'], 'player_id': perf['player_id'], 'rating_live': r})

live_df3 = pd.DataFrame(live_ratings_multi)
comp3 = df3.merge(live_df3, on=['match_id', 'player_id'], how='inner')

def corrected_pandas_predict_multi(row):
    mean, std = NS_QUALITY_STATS[row['group']]
    z = (row['corrected_non_scoring'] - mean) / std
    qs = GoalRescueLiveService.S_MIN + (1 - GoalRescueLiveService.S_MIN) * _sigmoid(GoalRescueLiveService.K * z)
    coeff = GoalRescueLiveService.COEFF_MULTIPLIER * BASE_COEFF[row['group']]
    new_goal = goal_pre_iso(row['goals'], row['shots'], coeff) * row['isolation'] * qs
    new_event = new_goal + row['assist_contribution']
    raw = row['processed'] * row['impact'] + new_event
    rating = live_service_multi._apply_sigmoid_transformation(raw_score=raw) - row['supremacy']
    return round(max(0.0, min(10.0, rating)), 1)

comp3['rating_corrected_pandas'] = comp3.apply(corrected_pandas_predict_multi, axis=1)
print(f"Matched {len(comp3)} performances across all positions\n")
print('Fidelity check per position (live code vs pandas, same corrected proxy + constants):')
for g in POSITIONS_ANALYSED:
    sub = comp3[comp3['group'] == g]
    diff = (sub['rating_live'] - sub['rating_corrected_pandas']).abs()
    print(f'  {g:8s} n={len(sub):4d}  max diff={diff.max():.6f}  n>1e-6={ (diff>1e-6).sum() }')

Matched 1102 performances across all positions

Fidelity check per position (live code vs pandas, same corrected proxy + constants):
  ST       n= 234  max diff=0.000000  n>1e-6=0
  Winger   n= 444  max diff=0.000000  n>1e-6=0
  CM       n= 424  max diff=0.000000  n>1e-6=0


**Result:** _[fill after running]_ - all three rows should read `max diff = 0.000000`, same as ST alone did. If Winger or CM show a nonzero diff, the override for that position doesn't match its source method exactly - check the mastery-bonus order and thresholds against the source before trusting anything downstream for that position.

### Sweeping a shared s_min across all three positions

Now that fidelity is proven (all three positions, one shared formula), the `s_min` sweep can run entirely in pandas on `comp3` - no need to re-run the match loop per candidate. `COEFF_MULTIPLIER` stays fixed at the previously validated 0.67 here; this sweep isolates `s_min` only. Read this as a place to eyeball a shared value, not an optimiser - Winger and CM don't have a stated target the way ST's ~7.0-7.2 does, since they were never the problem to begin with, so the goal is a shared `s_min` that lands ST where you want it **without** dragging Winger/CM lower than their own current, already-reasonable numbers.

In [104]:
S_MIN_CANDIDATES = [0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65]
K_FIXED = 1.5
COEFF_MULTIPLIER_FIXED = 0.67

def predict_for_smin(row, s_min):
    mean, std = NS_QUALITY_STATS[row['group']]
    z = (row['corrected_non_scoring'] - mean) / std
    qs = s_min + (1 - s_min) * _sigmoid(K_FIXED * z)
    coeff = COEFF_MULTIPLIER_FIXED * BASE_COEFF[row['group']]
    new_goal = goal_pre_iso(row['goals'], row['shots'], coeff) * row['isolation'] * qs
    new_event = new_goal + row['assist_contribution']
    raw = row['processed'] * row['impact'] + new_event
    rating = live_service_multi._apply_sigmoid_transformation(raw_score=raw) - row['supremacy']
    return round(max(0.0, min(10.0, rating)), 1)

sweep_rows = []
for s_min in S_MIN_CANDIDATES:
    row = {'s_min': s_min}
    for g in POSITIONS_ANALYSED:
        sub = comp3[(comp3['group'] == g) & comp3['corrected_bad_half'] & comp3['scored']]
        r = sub.apply(lambda x: predict_for_smin(x, s_min), axis=1)
        row[f'{g}_bad_scored'] = round(r.mean(), 2)
        sub_good = comp3[(comp3['group'] == g) & ~comp3['corrected_bad_half'] & comp3['scored']]
        r_good = sub_good.apply(lambda x: predict_for_smin(x, s_min), axis=1)
        row[f'{g}_good_scored'] = round(r_good.mean(), 2)
    sweep_rows.append(row)

print(f"Current (production, no fix) bad_scored / good_scored per position:")
for g in POSITIONS_ANALYSED:
    bs = comp3[(comp3['group']==g) & comp3['corrected_bad_half'] & comp3['scored']]['rating'].mean()
    gs = comp3[(comp3['group']==g) & ~comp3['corrected_bad_half'] & comp3['scored']]['rating'].mean()
    print(f'  {g:8s} bad_scored={bs:.2f}  good_scored={gs:.2f}')
print(f"\ns_min sweep (coeff_multiplier fixed at {COEFF_MULTIPLIER_FIXED}, k={K_FIXED}):")
print(pd.DataFrame(sweep_rows).set_index('s_min'))

Current (production, no fix) bad_scored / good_scored per position:
  ST       bad_scored=8.20  good_scored=8.82
  Winger   bad_scored=7.75  good_scored=8.67
  CM       bad_scored=7.68  good_scored=8.37

s_min sweep (coeff_multiplier fixed at 0.67, k=1.5):
       ST_bad_scored  ST_good_scored  Winger_bad_scored  Winger_good_scored  \
s_min                                                                         
0.35            6.96            8.37               6.70                8.19   
0.40            7.01            8.39               6.75                8.21   
0.45            7.07            8.40               6.80                8.21   
0.50            7.13            8.41               6.85                8.24   
0.55            7.19            8.42               6.89                8.25   
0.60            7.25            8.43               6.94                8.26   
0.65            7.30            8.44               6.97                8.27   

       CM_bad_scored  CM_good_s

**Result:** _[fill after running]_ - pick the `s_min` row where ST's `bad_scored` sits in your ~7.0-7.2 target while Winger and CM's `bad_scored` haven't dropped much further below their own current values (printed above the table) - since they weren't the problem, over-correcting them is the thing to avoid. If no single `s_min` satisfies both, that's a real finding: it means `coeff_multiplier` also needs to vary by position rather than staying at a flat 0.67 for everyone, which would be the natural next sweep.

## CM needs its own coefficient, not more s_min

`s_min` and the coefficient do different jobs. `s_min` is **conditional** - how much a goal is discounted for a *specifically poor* surrounding performance, within a position. Whether a CM's goal is inherently worth less than a striker's, regardless of how good the rest of the game was, is a **permanent, positional** question - that's what the coefficient is for. Tightening the shared `s_min` further would barely move CM's `good_scored` (it's already saturated, same pattern ST showed earlier) while over-correcting ST/Winger's `bad_scored`, which is currently landing where you want it.

So: hold `s_min=0.55` and `k=1.5` shared and fixed, keep ST/Winger's `coeff_multiplier` at the validated 0.67, and give **CM its own, separate** coefficient multiplier - sweeping it lower to see what actually brings `good_scored` down.

In [105]:
CM_COEFF_CANDIDATES = [0.67, 0.55, 0.45, 0.35, 0.25, 0.15]
S_MIN_FIXED = 0.55

def predict_general(row, s_min, coeff_mult_by_group):
    mean, std = NS_QUALITY_STATS[row['group']]
    z = (row['corrected_non_scoring'] - mean) / std
    qs = s_min + (1 - s_min) * _sigmoid(K_FIXED * z)
    coeff = coeff_mult_by_group[row['group']] * BASE_COEFF[row['group']]
    new_goal = goal_pre_iso(row['goals'], row['shots'], coeff) * row['isolation'] * qs
    new_event = new_goal + row['assist_contribution']
    raw = row['processed'] * row['impact'] + new_event
    rating = live_service_multi._apply_sigmoid_transformation(raw_score=raw) - row['supremacy']
    return round(max(0.0, min(10.0, rating)), 1)

cm_bad = comp3[(comp3['group'] == 'CM') & comp3['corrected_bad_half'] & comp3['scored']]
cm_good = comp3[(comp3['group'] == 'CM') & ~comp3['corrected_bad_half'] & comp3['scored']]

rows = []
for cm_mult in CM_COEFF_CANDIDATES:
    mults = {'ST': COEFF_MULTIPLIER_FIXED, 'Winger': COEFF_MULTIPLIER_FIXED, 'CM': cm_mult}
    bad_r = cm_bad.apply(lambda r: predict_general(r, S_MIN_FIXED, mults), axis=1)
    good_r = cm_good.apply(lambda r: predict_general(r, S_MIN_FIXED, mults), axis=1)
    rows.append({
        'cm_coeff_multiplier': cm_mult, 'effective_cm_goal_coeff': round(cm_mult * BASE_COEFF['CM'], 2),
        'bad_scored': round(bad_r.mean(), 2), 'good_scored': round(good_r.mean(), 2),
        'good_pct_7_5': round((good_r >= 7.5).mean() * 100, 0),
    })

print(f"CM current (production): bad_scored={cm_bad['rating'].mean():.2f}  good_scored={cm_good['rating'].mean():.2f}\n")
print(f"ST/Winger held at coeff_multiplier={COEFF_MULTIPLIER_FIXED}, s_min={S_MIN_FIXED} throughout - only CM's own coefficient varies:")
print(pd.DataFrame(rows).set_index('cm_coeff_multiplier'))

CM current (production): bad_scored=7.68  good_scored=8.37

ST/Winger held at coeff_multiplier=0.67, s_min=0.55 throughout - only CM's own coefficient varies:
                     effective_cm_goal_coeff  bad_scored  good_scored  \
cm_coeff_multiplier                                                     
0.67                                    0.67        6.92         7.99   
0.55                                    0.55        6.78         7.89   
0.45                                    0.45        6.68         7.78   
0.35                                    0.35        6.56         7.68   
0.25                                    0.25        6.44         7.58   
0.15                                    0.15        6.32         7.47   

                     good_pct_7_5  
cm_coeff_multiplier                
0.67                         71.0  
0.55                         69.0  
0.45                         66.0  
0.35                         63.0  
0.25                         54.0  
0.15

**Result:** _[fill after running]_ - watch two things moving in opposite directions as CM's coefficient drops: `good_scored` falls (the point of this sweep), but so does `bad_scored`, and `bad_scored` was never the complaint for CM. Pick the multiplier where `good_scored` reaches a level that feels right for a CM goal, then sanity-check `bad_scored` hasn't fallen further than makes sense - if it has, that's the real cost of treating CM goals as inherently worth less: it discounts *every* CM goal, not just the ones papering over a poor game. `effective_cm_goal_coeff` is the number that would actually replace CM's production `1.0` if you commit to a candidate.

## Locking in the final configuration

`CM coeff_multiplier = 0.3` (effective CM goal coefficient **0.30**, down from production's 1.0) - exact number, plus the full three-position picture in one table: `coeff_multiplier` 0.67 / 0.67 / 0.3 for ST / Winger / CM, shared `s_min=0.55`, `k=1.5`. This is the actual proposed configuration - everything below is what shipping it would look like.

In [106]:
FINAL_COEFF_MULTIPLIER = {'ST': 0.67, 'Winger': 0.67, 'CM': 0.30}
FINAL_S_MIN = 0.55

print('FINAL CONFIGURATION')
print(f"  s_min={FINAL_S_MIN}  k={K_FIXED}")
for g, m in FINAL_COEFF_MULTIPLIER.items():
    print(f"  {g:8s} coeff_multiplier={m}   effective goal coeff = {m * BASE_COEFF[g]:.3f} (was {BASE_COEFF[g]})")
print()

rows = []
for g in POSITIONS_ANALYSED:
    bad = comp3[(comp3['group'] == g) & comp3['corrected_bad_half'] & comp3['scored']]
    good = comp3[(comp3['group'] == g) & ~comp3['corrected_bad_half'] & comp3['scored']]
    bad_after = bad.apply(lambda r: predict_general(r, FINAL_S_MIN, FINAL_COEFF_MULTIPLIER), axis=1)
    good_after = good.apply(lambda r: predict_general(r, FINAL_S_MIN, FINAL_COEFF_MULTIPLIER), axis=1)
    rows.append({
        'position': g,
        'bad_scored_before': round(bad['rating'].mean(), 2), 'bad_scored_after': round(bad_after.mean(), 2),
        'good_scored_before': round(good['rating'].mean(), 2), 'good_scored_after': round(good_after.mean(), 2),
        'bad_pct7.5_before': round((bad['rating'] >= 7.5).mean()*100, 0),
        'bad_pct7.5_after': round((bad_after >= 7.5).mean()*100, 0),
        'good_pct7.5_before': round((good['rating'] >= 7.5).mean()*100, 0),
        'good_pct7.5_after': round((good_after >= 7.5).mean()*100, 0),
        'floor_after': round(bad_after.min(), 2),
    })

summary = pd.DataFrame(rows).set_index('position')
print('Before -> after, bad-but-scored and good-but-scored, all three positions:')
print(summary[['bad_scored_before', 'bad_scored_after', 'bad_pct7.5_before', 'bad_pct7.5_after']])
print()
print(summary[['good_scored_before', 'good_scored_after', 'good_pct7.5_before', 'good_pct7.5_after']])
print()
print('Floor (lowest bad-but-scored rating) under the final config:')
print(summary['floor_after'])

# Confirm non-scorers are still completely untouched under this final config too.
ns_before = comp3[~comp3['scored']]['rating']
ns_after = comp3[~comp3['scored']].apply(lambda r: predict_general(r, FINAL_S_MIN, FINAL_COEFF_MULTIPLIER), axis=1)
print(f"\nNon-scorer check: max |before-after| = {(ns_before - ns_after).abs().max():.6f} (should be 0.0)")

FINAL CONFIGURATION
  s_min=0.55  k=1.5
  ST       coeff_multiplier=0.67   effective goal coeff = 1.005 (was 1.5)
  Winger   coeff_multiplier=0.67   effective goal coeff = 0.871 (was 1.3)
  CM       coeff_multiplier=0.3   effective goal coeff = 0.300 (was 1.0)

Before -> after, bad-but-scored and good-but-scored, all three positions:
          bad_scored_before  bad_scored_after  bad_pct7.5_before  \
position                                                           
ST                     8.20              7.19               93.0   
Winger                 7.75              6.89               57.0   
CM                     7.68              6.50               56.0   

          bad_pct7.5_after  
position                    
ST                    29.0  
Winger                29.0  
CM                    12.0  

          good_scored_before  good_scored_after  good_pct7.5_before  \
position                                                              
ST                      8.82       

**Result:** _[fill after running]_ - this is the number that matters: the `bad_scored_after` / `good_scored_after` pair for CM at the exact 0.3 you picked, not an interpolation. Sanity-check the non-scorer line reads exactly `0.000000` - confirms the isolation property still holds even with a position-specific coefficient in the mix.

**If this table looks right, the configuration is decision-ready:** `coeff_multiplier` 0.67 / 0.67 / 0.3 for ST / Winger / CM (effective goal coefficients 1.005 / 0.871 / 0.30, down from production's 1.5 / 1.3 / 1.0), shared `s_min=0.55`, `k=1.5`, applied via the quality-scaled goal bonus proven correct against the live service for all three positions (0.0 diff). What's left before this becomes an actual PR: (1) turn the stand-in `NS_QUALITY_STATS` (computed in-sample from this save) into a real stored calibration constant - the same train/test question raised throughout this project; (2) decide whether CDM and Fullback need the same treatment (deferred per the notebook's original scope, but CDM in particular has its own inline goal-adjacent bonus worth a look); (3) write the actual patch to `match_ratings_service.py` mirroring these three overrides.

## ST and Winger, decoupled the same way CM was

With the corrected `XG_PER_SHOT`, ST's `good_scored` (8.42) and Winger's (8.25) both sit higher than what felt right before - and `s_min` has already proven it can't pull `good_scored` down further once it's saturated (same pattern that motivated CM's own coefficient). So this decouples ST and Winger from the shared 0.67 the same way CM was decoupled, each swept independently (they don't interact - each position's numbers depend only on its own rows) and both held against `s_min=0.55` and CM's already-settled `0.30`.

ST gets a wider range since it needs to come down "a good bit"; Winger a gentler one since it only needs to come down "a little".

In [107]:
ST_COEFF_CANDIDATES = [0.67, 0.55, 0.45, 0.35, 0.25]
WINGER_COEFF_CANDIDATES = [0.67, 0.60, 0.55, 0.50, 0.45]
S_MIN_FIXED2 = 0.55
CM_FIXED = 0.30

def sweep_position(position, candidates, other_fixed):
    bad = comp3[(comp3['group'] == position) & comp3['corrected_bad_half'] & comp3['scored']]
    good = comp3[(comp3['group'] == position) & ~comp3['corrected_bad_half'] & comp3['scored']]
    rows = []
    for mult in candidates:
        mults = dict(other_fixed)
        mults[position] = mult
        bad_r = bad.apply(lambda r: predict_general(r, S_MIN_FIXED2, mults), axis=1)
        good_r = good.apply(lambda r: predict_general(r, S_MIN_FIXED2, mults), axis=1)
        rows.append({
            f'{position.lower()}_coeff_multiplier': mult,
            'effective_goal_coeff': round(mult * BASE_COEFF[position], 3),
            'bad_scored': round(bad_r.mean(), 2), 'bad_pct_7_5': round((bad_r >= 7.5).mean()*100, 0),
            'good_scored': round(good_r.mean(), 2), 'good_pct_7_5': round((good_r >= 7.5).mean()*100, 0),
        })
    return pd.DataFrame(rows).set_index(f'{position.lower()}_coeff_multiplier')

print('ST current (production, no fix): bad_scored=8.20  good_scored=8.82')
print('ST sweep - Winger held at 0.67, CM held at 0.30:')
print(sweep_position('ST', ST_COEFF_CANDIDATES, {'Winger': 0.67, 'CM': CM_FIXED}))

print('\nWinger current (production, no fix): bad_scored=7.75  good_scored=8.67')
print('Winger sweep - ST held at 0.67, CM held at 0.30:')
print(sweep_position('Winger', WINGER_COEFF_CANDIDATES, {'ST': 0.67, 'CM': CM_FIXED}))

ST current (production, no fix): bad_scored=8.20  good_scored=8.82
ST sweep - Winger held at 0.67, CM held at 0.30:
                     effective_goal_coeff  bad_scored  bad_pct_7_5  \
st_coeff_multiplier                                                  
0.67                                1.005        7.19         29.0   
0.55                                0.825        7.00         25.0   
0.45                                0.675        6.83         18.0   
0.35                                0.525        6.65          4.0   
0.25                                0.375        6.47          4.0   

                     good_scored  good_pct_7_5  
st_coeff_multiplier                             
0.67                        8.42          78.0  
0.55                        8.29          72.0  
0.45                        8.17          69.0  
0.35                        8.03          66.0  
0.25                        7.89          60.0  

Winger current (production, no fix): bad_scored=7

**Result:** _[fill after running]_ - for ST, watch how far `good_scored` needs to fall to feel right, then check `bad_scored` hasn't dropped through the floor of your ~7.0-7.2 comfort zone in the process (the same tension CM showed - cutting the coefficient pulls both groups down, not just the one you're targeting). Winger only needs a small nudge, so the gentler candidate range should be enough - if `good_scored` is still too high even at the bottom of that range, widen it the same way ST's was.

## s_min needs to be per-position too

A single coefficient per position can't hit two independent targets (`bad_scored` AND `good_scored`) at once - that's exactly the wall just hit. The fix: hold each position's coefficient at the value that gets `good_scored` right (ST 0.25, Winger 0.45, CM stays 0.30 - unchanged, no complaint there), then sweep `s_min` **per position**, higher than the shared 0.55, to pull `bad_scored` back up. Since `s_min` only discounts low-quality goals, this should move `bad_scored` a lot while barely touching `good_scored` - the same saturation property already confirmed earlier, now used deliberately rather than fought against.

In [108]:
def predict_fully_general(row, s_min_by_group, coeff_mult_by_group, k=1.5):
    mean, std = NS_QUALITY_STATS[row['group']]
    z = (row['corrected_non_scoring'] - mean) / std
    s_min = s_min_by_group[row['group']]
    qs = s_min + (1 - s_min) * _sigmoid(k * z)
    coeff = coeff_mult_by_group[row['group']] * BASE_COEFF[row['group']]
    new_goal = goal_pre_iso(row['goals'], row['shots'], coeff) * row['isolation'] * qs
    new_event = new_goal + row['assist_contribution']
    raw = row['processed'] * row['impact'] + new_event
    rating = live_service_multi._apply_sigmoid_transformation(raw_score=raw) - row['supremacy']
    return round(max(0.0, min(10.0, rating)), 1)

# Coefficients now fixed at what you liked for good_scored; CM untouched.
LIKED_COEFF = {'ST': 0.25, 'Winger': 0.45, 'CM': 0.30}
S_MIN_CANDIDATES_HIGH = [0.55, 0.65, 0.75, 0.85, 0.95]

def sweep_smin_for_position(position, candidates):
    bad = comp3[(comp3['group'] == position) & comp3['corrected_bad_half'] & comp3['scored']]
    good = comp3[(comp3['group'] == position) & ~comp3['corrected_bad_half'] & comp3['scored']]
    rows = []
    for s_min in candidates:
        s_min_by_group = {'ST': 0.55, 'Winger': 0.55, 'CM': 0.55}
        s_min_by_group[position] = s_min
        bad_r = bad.apply(lambda r: predict_fully_general(r, s_min_by_group, LIKED_COEFF), axis=1)
        good_r = good.apply(lambda r: predict_fully_general(r, s_min_by_group, LIKED_COEFF), axis=1)
        rows.append({
            f'{position.lower()}_s_min': s_min,
            'bad_scored': round(bad_r.mean(), 2), 'bad_pct_7_5': round((bad_r >= 7.5).mean()*100, 0),
            'good_scored': round(good_r.mean(), 2), 'good_pct_7_5': round((good_r >= 7.5).mean()*100, 0),
        })
    return pd.DataFrame(rows).set_index(f'{position.lower()}_s_min')

print('ST @ coeff_multiplier=0.25 (liked good_scored) - sweeping s_min:')
print(sweep_smin_for_position('ST', S_MIN_CANDIDATES_HIGH))

print('\nWinger @ coeff_multiplier=0.45 (liked good_scored) - sweeping s_min:')
print(sweep_smin_for_position('Winger', S_MIN_CANDIDATES_HIGH))

ST @ coeff_multiplier=0.25 (liked good_scored) - sweeping s_min:
          bad_scored  bad_pct_7_5  good_scored  good_pct_7_5
st_s_min                                                    
0.55            6.47          4.0         7.89          60.0
0.65            6.51          4.0         7.91          61.0
0.75            6.58          4.0         7.91          63.0
0.85            6.62          4.0         7.92          63.0
0.95            6.66          7.0         7.93          63.0

Winger @ coeff_multiplier=0.45 (liked good_scored) - sweeping s_min:
              bad_scored  bad_pct_7_5  good_scored  good_pct_7_5
winger_s_min                                                    
0.55                6.61         22.0         8.00          67.0
0.65                6.68         24.0         8.02          69.0
0.75                6.74         25.0         8.03          69.0
0.85                6.80         25.0         8.06          71.0
0.95                6.86         27.0         8.

**Result:** _[fill after running]_ - for each position, find the `s_min` where `bad_scored` climbs back into a comfortable range while `good_scored` has barely moved from the 7.89 (ST) / 8.00 (Winger) you liked - if `good_scored` drifts more than a few hundredths as `s_min` rises, that's worth noting, but the saturation property predicts it shouldn't. Once both positions have a picked `(coeff_multiplier, s_min)` pair, the full per-position configuration is: ST `(0.25, ?)`, Winger `(0.45, ?)`, CM `(0.30, 0.55)` - genuinely final at that point, nothing left to re-derive.

## ST: the real (coeff, s_min) frontier

`s_min` alone can't get `bad_scored` to ~6.9 at `coeff=0.25` - as `s_min` approaches 1.0, the quality scalar approaches 1.0 for every performance, so the mechanism collapses toward a flat, unconditional coefficient cut. That's a hard ceiling on `bad_scored` for a given coefficient, not something more `s_min` can push past. Explicitly including `s_min=1.0` below shows that ceiling directly rather than extrapolating it from the trend.

So this is a genuine trade-off: a small grid across `coeff` (0.25-0.45) x `s_min` (up to 1.0) shows exactly what `good_scored` costs to buy `bad_scored` ~6.9, so the choice is visible rather than guessed at.

In [109]:
ST_COEFF_GRID = [0.25, 0.30, 0.35, 0.40, 0.45]
ST_SMIN_GRID = [0.55, 0.75, 0.85, 0.95, 1.0]

st_bad = comp3[(comp3['group'] == 'ST') & comp3['corrected_bad_half'] & comp3['scored']]
st_good = comp3[(comp3['group'] == 'ST') & ~comp3['corrected_bad_half'] & comp3['scored']]

rows = []
for coeff in ST_COEFF_GRID:
    for s_min in ST_SMIN_GRID:
        s_min_by_group = {'ST': s_min, 'Winger': 0.95, 'CM': 0.55}
        coeff_by_group = {'ST': coeff, 'Winger': 0.45, 'CM': 0.30}
        bad_r = st_bad.apply(lambda r: predict_fully_general(r, s_min_by_group, coeff_by_group), axis=1)
        good_r = st_good.apply(lambda r: predict_fully_general(r, s_min_by_group, coeff_by_group), axis=1)
        rows.append({
            'coeff_multiplier': coeff, 's_min': s_min,
            'bad_scored': round(bad_r.mean(), 2), 'good_scored': round(good_r.mean(), 2),
        })

grid = pd.DataFrame(rows)
pivot_bad = grid.pivot(index='coeff_multiplier', columns='s_min', values='bad_scored')
pivot_good = grid.pivot(index='coeff_multiplier', columns='s_min', values='good_scored')
print('bad_scored:')
print(pivot_bad)
print('\ngood_scored:')
print(pivot_good)

near_target = grid[grid['bad_scored'].between(6.85, 6.95)].sort_values('good_scored')
print('\nCombinations landing bad_scored in [6.85, 6.95], sorted by good_scored (lowest cost first):')
print(near_target.to_string(index=False))

bad_scored:
s_min             0.55  0.75  0.85  0.95  1.00
coeff_multiplier                              
0.25              6.47  6.58  6.62  6.66  6.68
0.30              6.56  6.68  6.74  6.79  6.81
0.35              6.65  6.79  6.84  6.91  6.94
0.40              6.73  6.89  6.97  7.02  7.06
0.45              6.83  6.99  7.08  7.15  7.17

good_scored:
s_min             0.55  0.75  0.85  0.95  1.00
coeff_multiplier                              
0.25              7.89  7.91  7.92  7.93  7.94
0.30              7.96  7.99  8.00  8.02  8.03
0.35              8.03  8.07  8.09  8.10  8.10
0.40              8.10  8.14  8.15  8.16  8.18
0.45              8.17  8.21  8.22  8.24  8.25

Combinations landing bad_scored in [6.85, 6.95], sorted by good_scored (lowest cost first):
 coeff_multiplier  s_min  bad_scored  good_scored
             0.35   0.95        6.91         8.10
             0.35   1.00        6.94         8.10
             0.40   0.75        6.89         8.14


**Result:** _[fill after running]_ - `s_min=1.0` in the `bad_scored` table is the literal ceiling for each coefficient, confirming the mechanism directly. The bottom table (`near_target`) is the actual decision: it lists every combination that gets `bad_scored` to ~6.9, ranked by how much `good_scored` it costs - pick the row with the lowest `good_scored` that still clears 6.9, since that's the cheapest way to buy the bad_scored you want.

## Final confirmation: the real live service, at the exact final numbers

Everything so far proved the *formula* matches live code (once, at shared defaults) and separately explored the *numbers* through the pandas formula. This closes the gap: a fresh live service with genuinely per-position `S_MIN`/`COEFF_MULTIPLIER`, set to the exact final values, run for real and diffed against the pandas prediction at those same values. If this reads `0.000000` like every other fidelity check in this notebook, ST/Winger/CM is genuinely, fully settled - not just formula-equivalent in principle.

In [110]:
FINAL_S_MIN = {'ST': 0.95, 'Winger': 0.95, 'CM': 0.55}
FINAL_COEFF_MULT = {'ST': 0.35, 'Winger': 0.45, 'CM': 0.30}

class GoalRescueLiveServiceFinal(MatchRatingsService):
    """The real, final, per-position configuration - not another sweep point."""
    NS_QUALITY_STATS = NS_QUALITY_STATS
    S_MIN_BY_GROUP = FINAL_S_MIN
    COEFF_MULT_BY_GROUP = FINAL_COEFF_MULT
    K = 1.5

    def _quality_scalar(self, group, non_scoring_quality):
        mean, std = self.NS_QUALITY_STATS[group]
        z = (non_scoring_quality - mean) / std
        s_min = self.S_MIN_BY_GROUP[group]
        return s_min + (1.0 - s_min) * _sigmoid(self.K * z)

    def _apply_st_modifiers(self, z_scores, final_weights, performance_metrics, isolation_multiplier=1.0):
        self._apply_z_score_floors(z_scores, {
            'tackles_p90_z': -0.5, 'tackle_success_rate_z': -0.5, 'possession_won_p90_z': -0.5,
            'goals_p90_z': -2.0, 'assists_p90_z': -2.0, 'non_goal_shots_p90_z': -2.0,
            'offsides_p90_z': -1.5,
        })
        raw_score = self._calculate_dot_product(z_scores=z_scores, weights=final_weights)
        goal_bonus_pre = self._effective_goal_bonus(
            goals=performance_metrics.get('goals', 0), shots=performance_metrics.get('shots', 0),
            coeff=self.COEFF_MULT_BY_GROUP['ST'] * BASE_COEFF['ST'])
        assist_bonus_pre = performance_metrics.get('assists', 0) * BASE_ASSIST_COEFF['ST']
        raw_score = self._apply_mastery_bonus(raw_score=raw_score, z_scores=z_scores,
                                               key_a='passes_p90_z', key_b='dribbles_p90_z', threshold=1.5, weight=0.25)
        raw_score = self._apply_st_black_hole_penalty(raw_score=raw_score, performance_metrics=performance_metrics)
        raw_score = self._apply_st_hold_up_bonus(raw_score=raw_score, performance_metrics=performance_metrics)
        raw_score = self._apply_st_wasteful_finisher_penalty(raw_score=raw_score, performance_metrics=performance_metrics)
        scoring_contrib = _scoring_dot_contribution(z_scores, final_weights)
        qs = self._quality_scalar('ST', raw_score - scoring_contrib)
        event_bonus = (goal_bonus_pre * qs + assist_bonus_pre) * isolation_multiplier
        return raw_score, event_bonus

    def _apply_winger_modifiers(self, z_scores, final_weights, performance_metrics, isolation_multiplier=1.0):
        self._apply_z_score_floors(z_scores, {
            'tackles_p90_z': -0.5, 'tackle_success_rate_z': -0.5, 'possession_won_p90_z': -0.5,
            'fouls_committed_p90_z': -1.5, 'possession_lost_p90_z': -1.5, 'offsides_p90_z': -2.0,
        })
        raw_score = self._calculate_dot_product(z_scores=z_scores, weights=final_weights)
        goal_bonus_pre = self._effective_goal_bonus(
            goals=performance_metrics.get('goals', 0), shots=performance_metrics.get('shots', 0),
            coeff=self.COEFF_MULT_BY_GROUP['Winger'] * BASE_COEFF['Winger'])
        assist_bonus_pre = performance_metrics.get('assists', 0) * BASE_ASSIST_COEFF['Winger']
        raw_score = self._apply_mastery_bonus(raw_score=raw_score, z_scores=z_scores,
                                               key_a='dribbles_p90_z', key_b='xt_bonus_p90_z', threshold=1.5, weight=0.25)
        raw_score = self._apply_mastery_bonus(raw_score=raw_score, z_scores=z_scores,
                                               key_a='passes_p90_z', key_b='xt_bonus_p90_z', threshold=1.5, weight=0.20)
        raw_score = self._apply_mastery_bonus(raw_score=raw_score, z_scores=z_scores,
                                               key_a='tackles_p90_z', key_b='possession_won_p90_z', threshold=1.0, weight=0.15)
        shots = performance_metrics.get('shots', 0)
        goals = performance_metrics.get('goals', 0)
        if (shots >= 3) and (goals == 0):
            raw_score -= (shots - 2) * 0.10
        scoring_contrib = _scoring_dot_contribution(z_scores, final_weights)
        qs = self._quality_scalar('Winger', raw_score - scoring_contrib)
        event_bonus = (goal_bonus_pre * qs + assist_bonus_pre) * isolation_multiplier
        return raw_score, event_bonus

    def _apply_cm_modifiers(self, z_scores, opponent_goals, final_weights, performance_metrics,
                             minutes_played, isolation_multiplier=1.0):
        raw_score = self._calculate_dot_product(z_scores=z_scores, weights=final_weights)
        goal_bonus_pre = self._effective_goal_bonus(
            goals=performance_metrics.get('goals', 0), shots=performance_metrics.get('shots', 0),
            coeff=self.COEFF_MULT_BY_GROUP['CM'] * BASE_COEFF['CM'])
        assist_bonus_pre = performance_metrics.get('assists', 0) * BASE_ASSIST_COEFF['CM']
        raw_score = self._apply_mastery_bonus(raw_score=raw_score, z_scores=z_scores,
                                               key_a='tackles_p90_z', key_b='possession_won_p90_z', threshold=1.5, weight=0.25)
        raw_score = self._apply_mastery_bonus(raw_score=raw_score, z_scores=z_scores,
                                               key_a='passes_p90_z', key_b='dribbles_p90_z', threshold=1.2, weight=0.25)
        raw_score = self._apply_cm_clean_sheet_bonus(raw_score=raw_score, opponent_goals=opponent_goals,
                                                      minutes_played=minutes_played)
        scoring_contrib = _scoring_dot_contribution(z_scores, final_weights)
        qs = self._quality_scalar('CM', raw_score - scoring_contrib)
        event_bonus = (goal_bonus_pre * qs + assist_bonus_pre) * isolation_multiplier
        return raw_score, event_bonus

final_live_service = GoalRescueLiveServiceFinal(weights, means_stds)
final_ratings = []
for match in data:
    mo = match['data']; hl = mo['half_length']
    for perf in match['player_performances']:
        if perf['performance_type'] != 'Outfield':
            continue
        positions = perf.get('positions_played', [])
        if len(positions) != 1:
            continue
        group = POSITION_GROUP_MAP.get(positions[0])
        if group not in POSITIONS_ANALYSED:
            continue
        r = final_live_service.calculate_outfield_rating(perf, mo, hl, TEAM_NAME)
        if r is not None:
            final_ratings.append({'match_id': match['id'], 'player_id': perf['player_id'], 'rating_final': r})

final_df = pd.DataFrame(final_ratings)
final_comp = comp3.merge(final_df, on=['match_id', 'player_id'], how='inner')
final_comp['rating_pandas_predicted'] = final_comp.apply(
    lambda r: predict_fully_general(r, FINAL_S_MIN, FINAL_COEFF_MULT), axis=1)

diff = (final_comp['rating_final'] - final_comp['rating_pandas_predicted']).abs()
print(f"Real live service vs pandas prediction, at the exact final config:")
print(f"  max diff={diff.max():.6f}  n>1e-6={ (diff>1e-6).sum() } / {len(final_comp)}")

print("\nFinal, live-confirmed group means:")
for g in POSITIONS_ANALYSED:
    bad = final_comp[(final_comp['group']==g) & final_comp['corrected_bad_half'] & final_comp['scored']]
    good = final_comp[(final_comp['group']==g) & ~final_comp['corrected_bad_half'] & final_comp['scored']]
    print(f"  {g:8s} bad_scored={bad['rating_final'].mean():.2f}  good_scored={good['rating_final'].mean():.2f}")

Real live service vs pandas prediction, at the exact final config:
  max diff=0.000000  n>1e-6=0 / 1102

Final, live-confirmed group means:
  ST       bad_scored=6.91  good_scored=8.10
  Winger   bad_scored=6.86  good_scored=8.08
  CM       bad_scored=6.50  good_scored=7.63


**Result:** _[fill after running]_ - `max diff` should read `0.000000`, same standard as every other check in this notebook. If it does, ST/Winger/CM is genuinely done: real code, exact final numbers, proven correct end to end.

## Worth checking before extending to CDM, CB, Fullback

Deferred from the start for the same reason CAM was excluded outright: these positions score far less often than ST/Winger/CM in a single save, so the sample backing any fix could be too thin to trust - exactly what this project's thin-cell checks exist to catch elsewhere. Cheap to check with the existing read-only capture (no new live-implementation work) before deciding whether the full sweep machinery is worth building for any of them.

In [111]:
EXTRA_POSITIONS = ['CDM', 'CB', 'Fullback']

service_v4 = GoalRescueCaptureServiceV3(weights, means_stds)
records_v4 = []
for match in data:
    mo = match['data']; hl = mo['half_length']
    is_home = mo.get('home_team_name') == TEAM_NAME
    team_xg = (mo.get('home_stats', {}) if is_home else mo.get('away_stats', {})).get('xg', 0)
    opp_xg  = (mo.get('away_stats', {}) if is_home else mo.get('home_stats', {})).get('xg', 0)
    supremacy = service_v4._calculate_match_supremacy_scalar(team_xg=team_xg, xg_against=opp_xg)
    for perf in match['player_performances']:
        if perf['performance_type'] != 'Outfield':
            continue
        positions = perf.get('positions_played', [])
        if len(positions) != 1:
            continue
        group = POSITION_GROUP_MAP.get(positions[0])
        if group not in EXTRA_POSITIONS:
            continue
        service_v4.reset_capture()
        rating = service_v4.calculate_outfield_rating(perf, mo, hl, TEAM_NAME)
        cap = service_v4.last_capture
        if rating is None or cap is None:
            continue
        records_v4.append({
            'group': group, 'rating': rating, 'goals': cap['goals'],
            'processed': cap['processed_raw_score'],
            'scoring_dot_contribution': cap['scoring_dot_contribution'],
        })

df4 = pd.DataFrame(records_v4)
df4['scored'] = df4['goals'] >= 1
df4['corrected_non_scoring'] = df4['processed'] - df4['scoring_dot_contribution']
df4['corrected_pctile'] = df4.groupby('group')['corrected_non_scoring'].rank(pct=True) * 100
df4['bad_half'] = df4['corrected_pctile'] < 50

print('Sample sizes and rescue gap, CDM/CB/Fullback (diagnostic only, no fix applied):')
for g in EXTRA_POSITIONS:
    sub = df4[df4['group'] == g]
    n_scored = int(sub['scored'].sum())
    bad = sub[sub['bad_half'] & sub['scored']]
    bad_not = sub[sub['bad_half'] & ~sub['scored']]
    print(f"\n{g}: n={len(sub)}, scored={n_scored} ({n_scored/len(sub)*100:.1f}%)")
    if len(bad) >= 5 and len(bad_not) >= 5:
        gap = bad['rating'].mean() - bad_not['rating'].mean()
        print(f"  bottom-half non-scoring: scored mean={bad['rating'].mean():.2f} (n={len(bad)})  "
              f"not-scored mean={bad_not['rating'].mean():.2f} (n={len(bad_not)})  gap={gap:+.2f}")
    else:
        print(f"  too few bottom-half performances to compare (scored n={len(bad)}, not-scored n={len(bad_not)}) - "
              f"same sparsity issue that excluded CAM")

Sample sizes and rescue gap, CDM/CB/Fullback (diagnostic only, no fix applied):

CDM: n=197, scored=6 (3.0%)
  too few bottom-half performances to compare (scored n=3, not-scored n=95) - same sparsity issue that excluded CAM

CB: n=414, scored=9 (2.2%)
  too few bottom-half performances to compare (scored n=3, not-scored n=203) - same sparsity issue that excluded CAM

Fullback: n=400, scored=0 (0.0%)
  too few bottom-half performances to compare (scored n=0, not-scored n=199) - same sparsity issue that excluded CAM


**Result:** _[fill after running]_ - the `scored` percentage and the bottom-half sample sizes are the actual decision. If a position's bottom-half-scored sample is single digits, that's the CAM situation again - not enough data to build anything trustworthy on, regardless of whether a rescue gap shows up. Only a position with both a decent sample AND a real gap is worth the full live-implementation-and-sweep treatment; the others can reasonably stay out of scope.

### Scope decision: CDM, CB, Fullback left unchanged

CDM (6 goals/197 perfs, 3 bottom-half), CB (9/414, 3 bottom-half), and Fullback (**0/400**) don't have a sample this investigation can draw any conclusion from - same sparsity that excluded CAM outright. **Deliberately leaving all three at their production goal coefficients** (`coeff_multiplier=1.0`, no quality scaling) rather than extrapolating the ST/Winger/CM-derived fix onto positions with no evidenced version of the problem: there's nothing here to validate a fix against, and at a 2-3% scoring rate (0% for Fullback) any latent version of the mechanism has negligible practical impact on these positions' ratings regardless.

**Revisit when:** the dataset grows enough for a workable bottom-half-scored sample - additional Valencia seasons, or pooling with the Arsenal FC 26 save once that's no longer being held out for OOD testing.

**Worth a cheap sanity check, not a full investigation:** Fullback's exact zero is stark enough to be worth eyeballing against `POSITION_GROUP_MAP` and the capture logic before trusting it as "fullbacks just don't score in this save" rather than a data issue.

## Caveats

- **`non_scoring_raw` is a proxy, not a clean isolation of non-scoring quality** -
  a goal leaks weakly via `non_goal_shots`. Same limitation scale-comparability
  carried; it biases *against* finding an effect (poor-but-scored performances get
  a small non-scoring bump from the goal), so any effect found is if anything
  understated.
- **This is one dominant Valencia save.** Poor-but-scored strikers on a dominant
  team also carry a supremacy deduction; on a struggling team the same
  performances could rate differently. Read every number as save-specific.
- **"Bad" is relative to this population's STs**, not an absolute standard.
  Bottom-half here means bottom-half among Valencia's single-position ST
  performances in this save.
- **The levers are re-ratings, not re-calibrations.** They show what these exact
  performances would have scored under a changed constant, holding everything else
  fixed. They do not re-fit anything or account for how a changed constant would
  interact with the rest of a full recalibration.
- **Lever C introduces a new coupling** (goal value now depends on non-scoring
  percentile, a population-relative quantity) that would need care to implement in
  a live, per-match service where the population isn't known at rating time - a
  design question, not handled here.
- **CAM excluded** for single-position data sparsity, though its 0.9 coefficient
  and generous floors make it a plausible candidate to show the same pattern.

## Conclusions

**1. The intuition holds, decisively.** A poor-non-scoring ST performance rates 5.86 if it doesn't score and 7.98 if it does (+2.12), with 72.4% of bad-but-scored performances at 7.5+ and the lowest rating any goal-scoring ST ever received being 6.80. The feeling that "a striker who plays badly but scores still gets a 7.5-8" is not a memory artefact - it is what the algorithm does.

**2. It is exactly the predicted mechanism.** The rescue gap tracks the goal coefficient (ST 1.5 > Winger 1.3 > CM 1.0); a cameo goal (<30 min, 7.86) is worth almost as much as a full-game goal (60+, 8.48) because the event bonus bypasses the minutes scalar; and for the bad-but-scored group the goal bonus is **93% of the raw score**. The goal isn't papering over the performance - as far as the rating sees it, the goal *is* the performance.

**3. What to change.** Of the levers: the **goal floor (B) is a dead end** (it only bites wasteful high-shot goals, which these mostly aren't). The **coefficient (A) works but is blunt** - it lowers every striker's goal equally, penalising the deserving 8s along with the undeserved ones. The **quality-scaled bonus (C) is the targeted fix** - it lowers a goal's value only when the surrounding play was poor - and it ships in-call via a stored per-position non-scoring mean/std plus a logistic, structurally the same idea as the impact scalar but pointed at the goal bonus. The collateral-cost comparison shows C leaving good scorers and all non-scorers untouched, which A cannot do.

**4. C vs the coeff-cut-plus-base-up-weight alternative.** They can land similarly *on scorers*, but differ in footprint: C touches only goals, whereas up-weighting the base re-scales every performance (including non-scorers). That base change is a legitimate but *separate* decision ("is good all-round play currently under-rewarded?"), not part of the goal-rescue fix. A combination - modest base up-weight for the separate concern, gentle C for this one - is defensible if you actually hold both concerns.

**5. Relationship to scale-comparability.** No contradiction: that notebook tested cross-position fairness at the *top* decile and found the attacking-vs-CM gap non-robust. This tested the *absolute, bottom-of-distribution* question it never asked, and found a large, mechanistically-explained effect. This is where the original "strikers feel overrated" intuition actually lived.

**Open before implementing C:** the three new constants (`s_min`, `k`, and the per-position non-scoring mean/std) are themselves hand-set - the exact thing 3.5 and the IRL-data grounding work should pin down rather than eyeball. And `non_scoring_raw` needs per-position standardisation (its weights don't sum to 1), which the stored mean/std handles but is real calibration work, not a constant swap.